# Colab 03 (Full): Step Inference + Evaluation (Inline)
- `inference_jssp_fssp.py` 핵심 옵션을 노트북형으로 전체화
- step-by-step 추론, reflection, rationale, demo/full-eval, csv 저장 포함
- strict parsing (fallback 없음), decoding masking 포함


In [1]:
!pip -q install -U unsloth tqdm
!pip -q install -U "transformers==4.57.6" huggingface_hub datasets accelerate bitsandbytes pandas matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

## 1) 설정


In [ ]:
import os
import re
import random
import csv
import time
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from huggingface_hub import login, hf_hub_download, HfApi, snapshot_download
from tqdm import tqdm
from unsloth import FastLanguageModel
import transformers
print('transformers version:', transformers.__version__)

import logging
import warnings
from transformers.utils import logging as hf_logging

def apply_transformers_logging_hotfix():
    """Avoid transformers 5.x warning_once logging format crash in Colab."""
    try:
        import transformers.modeling_attn_mask_utils as _amu
        if hasattr(_amu, 'logger') and hasattr(_amu.logger, 'warning_once'):
            _amu.logger.warning_once = lambda *args, **kwargs: None
    except Exception:
        pass

    hf_logging.set_verbosity_error()
    logging.getLogger('transformers').setLevel(logging.ERROR)
    warnings.filterwarnings(
        'ignore',
        message='The attention mask API under `transformers.modeling_attn_mask_utils`*',
    )

apply_transformers_logging_hotfix()
print('transformers logging hotfix enabled')

######################
# HUGGING FACE TOKEN #
######################
HF_TOKEN = ''
######################

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''
if not HF_TOKEN:
    print('HF_TOKEN is empty. Fill the block above or set a Colab secret/env var before Hugging Face operations.')
else:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF login ready')

CFG = {
    # model
    'model_type': 'llama8b',
    'model_repo_or_path': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2',
    'reason_model_repo_or_path': None,
    'enable_reason_adapter': False,
    # 'model_checkpoint_tag': 'checkpoint-33000',
    'model_checkpoint_tag': None,
    # 'reason_model_checkpoint_tag': 'checkpoint-33000
    'reason_model_checkpoint_tag': None,
    'model_path': None,
    'reason_model_path': None,
    'hf_token': HF_TOKEN,
    'max_seq_length': 16384,
    'dtype': 'bfloat16',
    'load_in_4bit': True,

    # eval dataset source
    'eval_source': 'hf',  # hf | local
    'eval_dataset_repo': 'HYUNJINI/jssp_validation_all_v1',
    'eval_dataset_file': 'validation_data/ta.json',
    'eval_data_path': '/content/ta.json',
    'infer_fssp': False,
    'use_distance': True,
    'train_vrp_tsp': False,
    'train_knapsack': False,
    'train_binpack': False,
    'train_jssp': False,
    'train_fssp': False,

    # environment mode
    'env_mode': 'dispatch',  # serial | dispatch

    # generation options
    'num_return_sequences': 10,
    'disable_masking': False,
    'step_action_max_new_tokens': 8,
    'disable_step_problem_context': False,
    'enable_step_improvement': True,
    'step_reflection_passes': 2,
    'step_reflection_topk': 5,
    'emit_step_rationale': False,
    'step_rationale_max_new_tokens': 96,
    'reason_topk': 3,
    'action_code_width': 4,
    'action_code_seed': 42,
    'action_code_cap': 9999,

    # run mode
    'demo_index': None,          # None 이면 전체 평가
    'demo_num_solutions': None,  # None이면 num_return_sequences 사용

    # output
    'output_accord': False,
    'output_list_of_lists': False,
    'output_dir': '/content/val_results/jssp_val',

    # optional debug
    'inspect_masking': False,
    'print_step_trace': True,
}
print(CFG)





🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
transformers version: 4.57.6
transformers logging hotfix enabled
HF login ready
{'model_type': 'llama8b', 'model_repo_or_path': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2', 'reason_model_repo_or_path': None, 'enable_reason_adapter': False, 'model_checkpoint_tag': 'checkpoint-33000', 'reason_model_checkpoint_tag': None, 'model_path': None, 'reason_model_path': None, 'hf_token': '', 'max_seq_length': 16384, 'dtype': 'bfloat16', 'load_in_4bit': True, 'eval_source': 'hf', 'eval_dataset_repo': 'HYUNJINI/jssp_validation_all_v1', 'eval_dataset_file': 'validation_data/ta.json', 'eval_data_path': '/content/ta.json', 'infer_fssp': False, 'use_distance': True, 'train_vrp_tsp': False, 'train_knapsack': False, 'train_binpack': False, 'train_jssp': False, 'train_fssp': False, 'env_mode': 'serial', 'num_return_sequences': 10, 'disable_masking': False, 'step_action_

## 2) 유틸/환경/마스킹 함수 (inline)


In [3]:
import copy

import csv

import os

import random

import re

import time

from functools import lru_cache

from dataclasses import dataclass

from pathlib import Path

from types import SimpleNamespace

from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

import torch

from tqdm import tqdm

def read_matrix_form_jssp(matrix_content: str, sep: str = ' '):
    lines = [ln.strip() for ln in matrix_content.strip().splitlines() if ln.strip()]
    n, m = map(int, lines[0].split(sep))
    inst_for_ortools = []
    for i in range(n):
        nums = list(map(int, lines[i + 1].split(sep)))
        if len(nums) != 2 * m:
            raise ValueError(f'row {i} has {len(nums)} ints, expected {2*m}')
        inst_for_ortools.append([[nums[2 * j], nums[2 * j + 1]] for j in range(m)])
    ms = None
    if len(lines) >= n + 2:
        try:
            ms = float(lines[n + 1])
        except Exception:
            ms = None
    return n, m, inst_for_ortools, ms

ACTION_TOKEN_PREFIX = "A"

LEGACY_ACTION_TOKEN_PREFIX = "S"

ACTION_CODE_PATTERN = re.compile(
    r"(?:Action\s*:\s*)?<\s*[aAsS]\s*(\d+)\s*>",
    re.IGNORECASE,
)

def format_action_code(
    code_index: int,
    code_width: int = 4,
    prefix: str = ACTION_TOKEN_PREFIX,
) -> str:
    if int(code_index) < 0:
        raise ValueError(f"code_index must be non-negative, got {code_index}.")
    if int(code_width) < 1:
        raise ValueError(f"code_width must be >= 1, got {code_width}.")
    return f"<{str(prefix)}{int(code_index):0{int(code_width)}d}>"

def parse_action_code(text: str, code_width: int = 4) -> Optional[str]:
    if not isinstance(text, str):
        return None
    match = ACTION_CODE_PATTERN.search(text)
    if not match:
        return None
    return format_action_code(int(match.group(1)), code_width=code_width)


def _safe_ratio(numerator: float, denominator: float) -> float:
    if float(denominator) == 0.0:
        return 0.0
    return float(numerator) / float(denominator)


def summarize_global_dynamic_state(state_json: Dict[str, object]) -> Dict[str, object]:
    machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
    current_cmax = int(
        state_json.get("current_cmax", max(machine_ready_time) if machine_ready_time else 0)
    )
    total_remaining_work = int(
        state_json.get("total_remaining_work", sum(state_json.get("remaining_work", [])))  # type: ignore[arg-type]
    )
    summary = {
        "step_idx": int(state_json["step_idx"]),
        "total_steps": int(state_json["total_steps"]),
        "scheduled_ratio": float(state_json.get("scheduled_ratio", 0.0)),
        "current_cmax": current_cmax,
        "total_remaining_work": total_remaining_work,
        "unfinished_jobs_count": int(state_json.get("unfinished_jobs_count", 0)),
        "unfinished_jobs_ratio": float(state_json.get("unfinished_jobs_ratio", 0.0)),
        "machine_ready_min": int(state_json.get("machine_ready_min", 0)),
        "machine_ready_mean": float(state_json.get("machine_ready_mean", 0.0)),
        "machine_ready_max": int(state_json.get("machine_ready_max", 0)),
        "machine_ready_std": float(state_json.get("machine_ready_std", 0.0)),
        "bottleneck_machine_id": int(state_json.get("bottleneck_machine_id", -1)),
        "bottleneck_machine_load": int(state_json.get("bottleneck_machine_load", 0)),
        "bottleneck_machine_ops_left": int(state_json.get("bottleneck_machine_ops_left", 0)),
    }
    return summary


def _sorted_reason_alternatives(
    action_effects: Sequence[Dict[str, object]],
    chosen_action_code: str,
    top_k: int = 3,
) -> List[Dict[str, object]]:
    alternatives = [
        dict(effect)
        for effect in action_effects
        if str(effect.get("action_code")) != str(chosen_action_code)
    ]
    alternatives.sort(
        key=lambda x: (
            int(x.get("estimated_makespan_after", 10**12)),
            int(x.get("delta_cmax", 10**12)),
            int(x.get("estimated_start", 10**12)),
            int(x.get("proc_time", 10**12)),
            int(x.get("job_id", 10**12)),
        )
    )
    return alternatives[: max(0, int(top_k))]


def _machine_label(machine_id: object) -> str:
    machine_int = int(machine_id)
    return f"M{machine_int}" if machine_int >= 0 else "M-"


def _format_value(value: object) -> str:
    if isinstance(value, float):
        return f"{value:.4f}".rstrip("0").rstrip(".")
    return str(value)


def _machine_token(machine_id: int) -> str:
    return f"M{int(machine_id)}" if int(machine_id) >= 0 else "M-"


def compute_action_transition_features(
    state_json: Dict[str, object],
    action_code_to_job: Dict[str, int],
) -> Tuple[int, List[Dict[str, object]]]:
    job_ready_time: List[int] = state_json["job_ready_time"]  # type: ignore[assignment]
    machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
    next_machine: List[int] = state_json["next_machine"]  # type: ignore[assignment]
    next_proc_time: List[int] = state_json["next_proc_time"]  # type: ignore[assignment]
    next2_machine: List[int] = state_json.get("next2_machine", [-1] * len(next_machine))  # type: ignore[assignment]
    next2_proc_time: List[int] = state_json.get("next2_proc_time", [0] * len(next_machine))  # type: ignore[assignment]
    remaining_ops: List[int] = state_json["remaining_ops"]  # type: ignore[assignment]
    remaining_work: List[int] = state_json["remaining_work"]  # type: ignore[assignment]
    job_total_ops: List[int] = state_json.get("job_total_ops", [1] * len(next_machine))  # type: ignore[assignment]
    job_total_work: List[int] = state_json.get("job_total_work", [1] * len(next_machine))  # type: ignore[assignment]
    machine_remaining_load: List[int] = state_json.get("machine_remaining_load", [0] * len(machine_ready_time))  # type: ignore[assignment]
    machine_remaining_ops: List[int] = state_json.get("machine_remaining_ops", [0] * len(machine_ready_time))  # type: ignore[assignment]
    post_route_tokens_by_job: List[List[str]] = state_json.get("post_route_tokens", [[] for _ in next_machine])  # type: ignore[assignment]

    current_cmax = int(
        state_json.get("current_cmax", max(machine_ready_time) if machine_ready_time else 0)
    )
    total_remaining_work = int(
        state_json.get("total_remaining_work", sum(int(x) for x in remaining_work))
    )

    effects: List[Dict[str, object]] = []
    for action_code, job_id in action_code_to_job.items():
        job = int(job_id)
        machine_id = int(next_machine[job])
        proc_time = int(next_proc_time[job])
        if machine_id < 0 or proc_time <= 0:
            continue

        job_ready_before = int(job_ready_time[job])
        machine_ready_before = int(machine_ready_time[machine_id])
        est_start = max(job_ready_before, machine_ready_before)
        est_end = est_start + proc_time
        current_cmax_after = max(current_cmax, est_end)
        delta_cmax = current_cmax_after - current_cmax
        rem_ops_before = int(remaining_ops[job])
        rem_ops_after = max(0, rem_ops_before - 1)
        rem_work_before = int(remaining_work[job])
        rem_work_after = max(0, rem_work_before - proc_time)
        total_ops = max(int(job_total_ops[job]), 1)
        total_work = max(int(job_total_work[job]), 1)
        job_progress_ratio_before = float(total_ops - rem_ops_before) / float(total_ops)
        job_progress_ratio_after = float(total_ops - rem_ops_after) / float(total_ops)
        affected_machine_load = int(machine_remaining_load[machine_id])
        affected_machine_ops_left = int(machine_remaining_ops[machine_id])

        effect = {
            "action_code": str(action_code),
            "job_id": job,
            "machine_id": machine_id,
            "machine_token": _machine_token(machine_id),
            "proc_time": proc_time,
            "next_machine": machine_id,
            "next_proc_time": proc_time,
            "next2_machine": int(next2_machine[job]),
            "next2_proc_time": int(next2_proc_time[job]),
            "remaining_ops_before": rem_ops_before,
            "remaining_ops_after": rem_ops_after,
            "remaining_work_before": rem_work_before,
            "remaining_work_after": rem_work_after,
            "job_progress_ratio_before": float(job_progress_ratio_before),
            "job_progress_ratio_after": float(job_progress_ratio_after),
            "job_ready_before": job_ready_before,
            "job_ready_after": int(est_end),
            "machine_ready_before": machine_ready_before,
            "machine_ready_after": int(est_end),
            "estimated_start": int(est_start),
            "estimated_end": int(est_end),
            "est_start": int(est_start),
            "est_end": int(est_end),
            "current_cmax_before": int(current_cmax),
            "current_cmax_after": int(current_cmax_after),
            "estimated_makespan_after": int(current_cmax_after),
            "delta_cmax": int(delta_cmax),
            "delta_cmax_ratio": float(_safe_ratio(delta_cmax, max(current_cmax_after, 1))),
            "job_wait": int(max(0, machine_ready_before - job_ready_before)),
            "machine_idle_gap": int(max(0, job_ready_before - machine_ready_before)),
            "slack_to_current_cmax": int(current_cmax - est_end),
            "affected_machine_load": affected_machine_load,
            "affected_machine_ops_left": affected_machine_ops_left,
            "affected_machine_load_ratio": float(
                _safe_ratio(affected_machine_load, max(total_remaining_work, 1))
            ),
            "remaining_work_after_ratio": float(_safe_ratio(rem_work_after, total_work)),
            "post_route_tokens": list(post_route_tokens_by_job[job]),
            "post_route_len": int(len(post_route_tokens_by_job[job])),
        }
        effects.append(effect)

    effects.sort(
        key=lambda x: (
            int(x["estimated_makespan_after"]),
            int(x["estimated_start"]),
            int(x["proc_time"]),
            int(x["job_id"]),
        )
    )
    return int(current_cmax), effects


def render_action_transition_line(effect: Dict[str, object]) -> str:
    return (
        f"{effect['action_code']} | "
        f"operation machine={_machine_token(int(effect['next_machine']))} | "
        f"processing time={effect['next_proc_time']} | "
        f"rem_ops:{effect['remaining_ops_before']}->{effect['remaining_ops_after']} | "
        f"rem_work:{effect['remaining_work_before']}->{effect['remaining_work_after']} | "
        f"job_prog:{_format_value(effect['job_progress_ratio_before'])}->{_format_value(effect['job_progress_ratio_after'])} | "
        f"job_ready:{effect['job_ready_before']}->{effect['job_ready_after']} | "
        f"machine_ready:{effect['machine_ready_before']}->{effect['machine_ready_after']} | "
        f"est_start={effect['estimated_start']} | "
        f"est_end={effect['estimated_end']} | "
        f"cmax:{effect['current_cmax_before']}->{effect['current_cmax_after']} | "
        f"delta_cmax={effect['delta_cmax']} | "
        f"delta_cmax_ratio={_format_value(effect['delta_cmax_ratio'])} | "
        f"job_wait={effect['job_wait']} | "
        f"machine_idle_gap={effect['machine_idle_gap']} | "
        f"slack_to_cmax={effect['slack_to_current_cmax']} | "
        f"machine_load={effect['affected_machine_load']} | "
        f"machine_ops_left={effect['affected_machine_ops_left']} | "
        f"machine_load_ratio={_format_value(effect['affected_machine_load_ratio'])} | "
        f"rem_work_after_ratio={_format_value(effect['remaining_work_after_ratio'])} | "
        f"post_route={_format_route_tokens(effect.get('post_route_tokens', []))}"
    )


def _order_prompt_effects_randomly(
    state_json: Dict[str, object],
    action_effects: Sequence[Dict[str, object]],
) -> List[Dict[str, object]]:
    ordered_effects = [dict(effect) for effect in action_effects]
    seed_material = (
        f"{int(state_json.get('step_idx', 0))}|"
        f"{int(state_json.get('current_cmax', 0))}|"
        + "|".join(sorted(str(effect.get("action_code", "")) for effect in ordered_effects))
    )
    rng = random.Random(seed_material)
    rng.shuffle(ordered_effects)
    return ordered_effects


def build_step_prompt(
    state_json: Dict[str, object],
    feasible_jobs: Sequence[int],
    step_idx: int,
    total_steps: int,
    problem_context_text: Optional[str] = None,
    action_code_to_job: Optional[Dict[str, int]] = None,
) -> str:
    """
    Build deterministic compact prompt for one-step action selection.

    Output format is always one line.
    - Default (legacy): Action: Job <id>
    - Action-token mode: <Axxxx>
    """
    lines = [
        "You are solving JSSP step-by-step.",
        "Objective: Minimize final makespan (Cmax) while keeping every action feasible.",
    ]
    if problem_context_text:
        lines.extend(
            [
                "Static problem context:",
                problem_context_text,
            ]
        )

    global_summary = summarize_global_dynamic_state(state_json)
    lines.extend(
        [
            "Global dynamic state:",
            (
                f"step={int(global_summary['step_idx']) + 1}/{int(global_summary['total_steps'])} "
                f"scheduled_ratio={_format_value(global_summary['scheduled_ratio'])}"
            ),
            f"current_cmax={global_summary['current_cmax']}",
            f"total_remaining_work={global_summary['total_remaining_work']}",
            (
                f"unfinished_jobs_count={global_summary['unfinished_jobs_count']} "
                f"unfinished_jobs_ratio={_format_value(global_summary['unfinished_jobs_ratio'])}"
            ),
            (
                f"machine_ready_min={global_summary['machine_ready_min']} "
                f"machine_ready_mean={_format_value(global_summary['machine_ready_mean'])} "
                f"machine_ready_max={global_summary['machine_ready_max']} "
                f"machine_ready_std={_format_value(global_summary['machine_ready_std'])}"
            ),
            (
                f"bottleneck_machine={_machine_token(int(global_summary['bottleneck_machine_id']))} "
                f"bottleneck_load={global_summary['bottleneck_machine_load']} "
                f"bottleneck_ops_left={global_summary['bottleneck_machine_ops_left']}"
            ),
        ]
    )

    if action_code_to_job:
        action_codes = list(action_code_to_job.keys())
        _, action_effects = compute_action_transition_features(
            state_json=state_json,
            action_code_to_job=action_code_to_job,
        )
        lines.extend(
            [
                f"Feasible action codes: {_format_action_codes(action_codes)}",
                "Candidate transition features:",
            ]
        )
        prompt_effects = _order_prompt_effects_randomly(state_json, action_effects)
        for effect in prompt_effects:
            lines.append(render_action_transition_line(effect))
        lines.extend(
            [
                "Action codes are randomized at each step. Do not assume persistent code identity.",
                "Choose exactly one feasible action code.",
                "Return exactly one code from the feasible action set, for example <A3812>.",
            ]
        )
    else:
        job_next_op: List[int] = state_json["job_next_op"]  # type: ignore[assignment]
        next_machine: List[int] = state_json["next_machine"]  # type: ignore[assignment]
        next_proc_time: List[int] = state_json["next_proc_time"]  # type: ignore[assignment]
        job_ready_time: List[int] = state_json["job_ready_time"]  # type: ignore[assignment]
        remaining_ops: List[int] = state_json["remaining_ops"]  # type: ignore[assignment]
        remaining_work: List[int] = state_json["remaining_work"]  # type: ignore[assignment]
        machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
        lines.extend(
            [
                f"Feasible jobs: {_format_feasible_jobs(feasible_jobs)}",
                "Job state summary:",
            ]
        )
        for job_id in range(len(job_next_op)):
            lines.append(
                (
                    f"Job {job_id}: "
                    f"next_op={job_next_op[job_id]}, "
                    f"next_machine={next_machine[job_id]}, "
                    f"next_proc={next_proc_time[job_id]}, "
                    f"job_ready={job_ready_time[job_id]}, "
                    f"remaining_ops={remaining_ops[job_id]}, "
                    f"remaining_work={remaining_work[job_id]}"
                )
            )
        lines.extend(
            [
                f"Machine ready times: {_format_machine_ready(machine_ready_time)}",
                "Choose exactly one job from feasible jobs to minimize final makespan (Cmax).",
                "Return exactly one line: Action: Job <id>",
            ]
        )

    return "\n".join(lines)


def build_randomized_action_code_map(
    feasible_jobs: Sequence[int],
    rng: Optional[random.Random] = None,
    code_width: int = 4,
    code_start: int = 1,
    code_cap: int = 9999,
) -> Dict[str, int]:
    jobs = [int(j) for j in feasible_jobs]
    if len(set(jobs)) != len(jobs):
        raise ValueError(f"feasible_jobs contains duplicates: {jobs}")
    if code_start < 0:
        raise ValueError(f"code_start must be non-negative, got {code_start}")
    if code_cap < code_start:
        raise ValueError(
            f"code_cap must be >= code_start, got code_start={code_start}, code_cap={code_cap}"
        )
    if len(jobs) > (int(code_cap) - int(code_start) + 1):
        raise ValueError(
            "Not enough action-code slots for this step: "
            f"jobs={len(jobs)}, available={int(code_cap) - int(code_start) + 1}, "
            f"range=[{code_start}, {code_cap}]"
        )
    if not jobs:
        return {}

    shuffled_jobs = list(jobs)
    if rng is None:
        random.shuffle(shuffled_jobs)
        sampled_code_indices = random.sample(
            range(int(code_start), int(code_cap) + 1), k=len(shuffled_jobs)
        )
    else:
        rng.shuffle(shuffled_jobs)
        sampled_code_indices = rng.sample(
            range(int(code_start), int(code_cap) + 1), k=len(shuffled_jobs)
        )

    action_code_to_job: Dict[str, int] = {}
    for code_idx, job_id in zip(sampled_code_indices, shuffled_jobs):
        code = format_action_code(int(code_idx), code_width=code_width)
        action_code_to_job[code] = int(job_id)
    return action_code_to_job


def invert_action_code_map(action_code_to_job: Dict[str, int]) -> Dict[int, str]:
    job_to_action_code: Dict[int, str] = {}
    for code, job_id in action_code_to_job.items():
        if job_id in job_to_action_code:
            raise ValueError(f"Duplicate job id in action_code_to_job: {job_id}")
        job_to_action_code[int(job_id)] = str(code)
    return job_to_action_code


def action_code_to_token_id(tokenizer, action_code: str) -> int:
    token_id = tokenizer.convert_tokens_to_ids(str(action_code))
    if token_id is None:
        raise ValueError(f"Tokenizer returned None token id for action_code={action_code!r}")
    token_id = int(token_id)
    unk_id = getattr(tokenizer, "unk_token_id", None)
    if unk_id is not None and token_id == int(unk_id):
        encoded = tokenizer.encode(str(action_code), add_special_tokens=False)
        if len(encoded) == 1:
            return int(encoded[0])
        raise ValueError(
            f"Action code {action_code!r} is not a single tokenizer token. "
            "Special tokens were likely not installed correctly."
        )
    return token_id


def token_id_to_action_code(tokenizer, token_id: int, code_width: int = 4):
    token = tokenizer.convert_ids_to_tokens(int(token_id))
    if isinstance(token, bytes):
        token = token.decode("utf-8", errors="ignore")
    token_text = str(token) if token is not None else ""
    parsed = parse_action_code(token_text, code_width=code_width)
    if parsed is not None:
        return parsed
    decoded = tokenizer.decode([int(token_id)], skip_special_tokens=False)
    return parse_action_code(decoded, code_width=code_width)


def build_problem_context_text(inst_for_ortools: Sequence[Sequence[Sequence[int]]]) -> str:
    """
    Build minimal static problem context text.
    """
    num_jobs = len(inst_for_ortools)
    total_ops = sum(len(job_ops) for job_ops in inst_for_ortools)
    max_machine = -1
    for job_ops in inst_for_ortools:
        for machine_id, _ in job_ops:
            max_machine = max(max_machine, int(machine_id))
    num_machines = max_machine + 1 if max_machine >= 0 else 0

    return f"Problem: {num_jobs} jobs x {num_machines} machines (total_ops={total_ops})"

class StepActionParseError(ValueError):
    """Raised when a generated step action cannot be parsed."""

def _normalize_text(text: str) -> str:
    return text.replace("\r", "")

def _is_prefix_of_any(candidate_text: str, feasible_action_codes: Sequence[str]) -> bool:
    candidate_text = _normalize_text(candidate_text)
    for code in feasible_action_codes:
        if code.startswith(candidate_text):
            return True
    return False

def build_step_prefix_allowed_tokens_fn(
    tokenizer,
    feasible_action_codes_provider: Iterable[str],
    prompt_len: int = 0,
    code_width: int = 4,
    code_cap: int = 9999,
):
    feasible_action_codes = tuple(str(code) for code in feasible_action_codes_provider)
    if not feasible_action_codes:
        raise RuntimeError("No feasible action codes available at this step.")

    feasible_token_ids = []
    for action_code in feasible_action_codes:
        token_id = tokenizer.convert_tokens_to_ids(str(action_code))
        if token_id is None:
            raise RuntimeError(f"Tokenizer returned None token id for action_code={action_code!r}")
        token_id = int(token_id)
        unk_id = getattr(tokenizer, 'unk_token_id', None)
        if unk_id is not None and token_id == int(unk_id):
            encoded = tokenizer.encode(str(action_code), add_special_tokens=False)
            if len(encoded) != 1:
                raise RuntimeError(
                    f"Action code {action_code!r} is not a single tokenizer token."
                )
            token_id = int(encoded[0])
        feasible_token_ids.append(token_id)
    feasible_token_ids = tuple(sorted(set(feasible_token_ids)))
    eos_token_id = getattr(tokenizer, "eos_token_id", None)

    def prefix_allowed_tokens_fn(batch_id: int, input_ids) -> List[int]:
        if hasattr(input_ids, "tolist"):
            input_ids = input_ids.tolist()
        generated_ids = input_ids[int(prompt_len) :]
        if len(generated_ids) == 0:
            return list(feasible_token_ids)

        chosen_action_code = token_id_to_action_code(
            tokenizer,
            int(generated_ids[0]),
            code_width=code_width,
        )
        if (
            len(generated_ids) == 1
            and chosen_action_code is not None
            and str(chosen_action_code) in feasible_action_codes
            and eos_token_id is not None
        ):
            return [int(eos_token_id)]

        if len(generated_ids) == 1 and chosen_action_code is not None and str(chosen_action_code) in feasible_action_codes:
            return list(feasible_token_ids)

        generated_text = _normalize_text(
            tokenizer.decode(generated_ids, skip_special_tokens=False)
        )
        raise RuntimeError(
            "No valid next tokens for step action generation. "
            f"generated_text={generated_text!r}, feasible_action_codes={list(feasible_action_codes)}"
        )

    return prefix_allowed_tokens_fn

HEADER_PATTERN = re.compile(
    r"JSSP\s+with\s+(\d+)\s+Jobs,\s*(\d+)\s+Machines",
    re.IGNORECASE,
)

JOB_HEADER_PATTERN = re.compile(
    r"^\s*Job\s+(\d+)\s+consists\s+of\s+Operations:\s*$",
    re.IGNORECASE,
)

OPERATION_PATTERN = re.compile(
    r"^\s*Operation\s+(\d+):\s*M(\d+),\s*(\d+)\s*$",
    re.IGNORECASE,
)

SOLUTION_OPERATION_PATTERN = re.compile(
    r"Job\s*(\d+)\s*Operation\s*(\d+),\s*M(\d+)",
    re.IGNORECASE,
)

MAKESPAN_PATTERN = re.compile(r"Makespan\s*:\s*(\d+)", re.IGNORECASE)

@dataclass(frozen=True)
class ParsedTeacherAction:
    """Parsed action from one-shot output text."""

    job_id: int
    op_idx: int
    machine_id: int

def parse_prompt_jobs_first(prompt_text: str, strict: bool = True) -> List[List[List[int]]]:
    """
    Parse `prompt_jobs_first` text into `inst_for_ortools` format.

    Returns:
        inst_for_ortools[job][op] = [machine, duration]
    """
    if not isinstance(prompt_text, str) or not prompt_text.strip():
        raise ValueError("prompt_text must be a non-empty string.")

    header_match = HEADER_PATTERN.search(prompt_text)
    expected_jobs = int(header_match.group(1)) if header_match else None
    expected_machines = int(header_match.group(2)) if header_match else None

    jobs: Dict[int, List[List[int]]] = {}
    current_job: Optional[int] = None

    for raw_line in prompt_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        job_match = JOB_HEADER_PATTERN.match(line)
        if job_match:
            current_job = int(job_match.group(1))
            jobs.setdefault(current_job, [])
            continue

        op_match = OPERATION_PATTERN.match(line)
        if op_match:
            if current_job is None:
                raise ValueError(f"Operation line appears before job header: {raw_line!r}")

            op_idx = int(op_match.group(1))
            machine = int(op_match.group(2))
            duration = int(op_match.group(3))
            expected_op_idx = len(jobs[current_job])
            if strict and op_idx != expected_op_idx:
                raise ValueError(
                    f"Operation index mismatch in Job {current_job}: "
                    f"expected {expected_op_idx}, got {op_idx}."
                )
            jobs[current_job].append([machine, duration])

    if not jobs:
        raise ValueError("Failed to parse any jobs from prompt_text.")

    max_job_id = max(jobs.keys())
    if strict and set(jobs.keys()) != set(range(max_job_id + 1)):
        raise ValueError("Job IDs must be contiguous from 0 when strict=True.")

    inst_for_ortools = [jobs[job_id] for job_id in range(max_job_id + 1)]

    if strict:
        if expected_jobs is not None and expected_jobs != len(inst_for_ortools):
            raise ValueError(
                f"Header jobs ({expected_jobs}) != parsed jobs ({len(inst_for_ortools)})."
            )
        if expected_machines is not None:
            for job_id, job_ops in enumerate(inst_for_ortools):
                if len(job_ops) != expected_machines:
                    raise ValueError(
                        f"Job {job_id} has {len(job_ops)} ops, expected {expected_machines}."
                    )

    return inst_for_ortools

def parse_solution_actions(
    solution_text: str, strict: bool = True
) -> Tuple[List[ParsedTeacherAction], Optional[int]]:
    """
    Parse one-shot output into per-step teacher actions.

    Returns:
        actions in decode order, declared makespan from text if present.
    """
    if not isinstance(solution_text, str) or not solution_text.strip():
        raise ValueError("solution_text must be a non-empty string.")

    matches = SOLUTION_OPERATION_PATTERN.findall(solution_text)
    if not matches:
        raise ValueError("No 'Job X Operation Y, MZ' actions found in solution_text.")

    next_expected_op: Dict[int, int] = {}
    actions: List[ParsedTeacherAction] = []
    for job_str, op_str, machine_str in matches:
        job_id = int(job_str)
        op_idx = int(op_str)
        machine_id = int(machine_str)

        expected_op_idx = next_expected_op.get(job_id, 0)
        if strict and op_idx != expected_op_idx:
            raise ValueError(
                f"Teacher action order violation for job {job_id}: "
                f"expected op {expected_op_idx}, got {op_idx}."
            )

        next_expected_op[job_id] = op_idx + 1
        actions.append(ParsedTeacherAction(job_id=job_id, op_idx=op_idx, machine_id=machine_id))

    makespan_match = MAKESPAN_PATTERN.search(solution_text)
    declared_makespan = int(makespan_match.group(1)) if makespan_match else None
    return actions, declared_makespan

class StaticJSSPStepEnv:
    """
    Deterministic static JSSP environment for step-by-step action selection.

    Action:
        choose one feasible job id at each step.
    """

    def __init__(self, inst_for_ortools: Sequence[Sequence[Sequence[int]]]):
        if not inst_for_ortools:
            raise ValueError("inst_for_ortools must not be empty.")

        self.inst_for_ortools: List[List[Tuple[int, int]]] = []
        for job_ops in inst_for_ortools:
            parsed_job_ops: List[Tuple[int, int]] = []
            for op in job_ops:
                if len(op) != 2:
                    raise ValueError(f"Each operation must be [machine, duration], got {op}.")
                machine, duration = int(op[0]), int(op[1])
                parsed_job_ops.append((machine, duration))
            self.inst_for_ortools.append(parsed_job_ops)

        self.num_jobs = len(self.inst_for_ortools)
        self.operations_per_job = [len(job_ops) for job_ops in self.inst_for_ortools]
        self.job_total_ops = list(self.operations_per_job)
        self.job_total_work = [
            sum(duration for _, duration in job_ops) for job_ops in self.inst_for_ortools
        ]
        self.total_ops = sum(self.operations_per_job)

        if self.total_ops <= 0:
            raise ValueError("total_ops must be positive.")

        max_machine_id = max(
            machine for job_ops in self.inst_for_ortools for machine, _ in job_ops
        )
        self.num_machines = max_machine_id + 1

        self.job_next_op: List[int] = []
        self.job_ready_time: List[int] = []
        self.machine_ready_time: List[int] = []
        self.scheduled_ops = 0
        self.event_log: List[Dict[str, int]] = []
        self.reset()

    @classmethod
    def from_prompt_jobs_first(cls, prompt_text: str, strict: bool = True) -> "StaticJSSPStepEnv":
        inst_for_ortools = parse_prompt_jobs_first(prompt_text, strict=strict)
        return cls(inst_for_ortools)

    def reset(self) -> Dict[str, object]:
        self.job_next_op = [0] * self.num_jobs
        self.job_ready_time = [0] * self.num_jobs
        self.machine_ready_time = [0] * self.num_machines
        self.scheduled_ops = 0
        self.event_log = []
        return self.get_state_json()

    def is_done(self) -> bool:
        return self.scheduled_ops == self.total_ops

    def get_makespan(self) -> int:
        return max(self.machine_ready_time) if self.machine_ready_time else 0

    def get_feasible_jobs(self) -> List[int]:
        return [
            job_id
            for job_id in range(self.num_jobs)
            if self.job_next_op[job_id] < self.operations_per_job[job_id]
        ]

    def _remaining_work(self, job_id: int) -> int:
        next_op = self.job_next_op[job_id]
        return sum(duration for _, duration in self.inst_for_ortools[job_id][next_op:])

    def _post_route_tokens(self, job_id: int) -> List[str]:
        next_op = self.job_next_op[job_id] + 1
        return [
            f"M{int(machine_id)}:{int(duration)}"
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]
        ]

    def _next_operation(self, job_id: int, offset: int = 0) -> Tuple[int, int]:
        op_idx = self.job_next_op[job_id] + int(offset)
        if op_idx >= self.operations_per_job[job_id]:
            return -1, 0
        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        return int(machine_id), int(duration)

    def _remaining_machine_loads_and_ops(self) -> Tuple[List[int], List[int]]:
        machine_remaining_load = [0] * self.num_machines
        machine_remaining_ops = [0] * self.num_machines

        for job_id in range(self.num_jobs):
            next_op = self.job_next_op[job_id]
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]:
                machine_remaining_load[int(machine_id)] += int(duration)
                machine_remaining_ops[int(machine_id)] += 1

        return machine_remaining_load, machine_remaining_ops

    def get_state_json(self) -> Dict[str, object]:
        next_machine: List[int] = []
        next_proc_time: List[int] = []
        next2_machine: List[int] = []
        next2_proc_time: List[int] = []
        remaining_ops: List[int] = []
        remaining_work: List[int] = []
        remaining_work_ratio: List[float] = []
        job_progress_ratio: List[float] = []
        post_route_tokens: List[List[str]] = []

        for job_id in range(self.num_jobs):
            op_idx = self.job_next_op[job_id]
            machine, duration = self._next_operation(job_id, offset=0)
            machine2, duration2 = self._next_operation(job_id, offset=1)
            next_machine.append(machine)
            next_proc_time.append(duration)
            next2_machine.append(machine2)
            next2_proc_time.append(duration2)

            rem_ops = self.operations_per_job[job_id] - op_idx
            rem_work = self._remaining_work(job_id)
            total_work = max(int(self.job_total_work[job_id]), 1)
            total_ops = max(int(self.job_total_ops[job_id]), 1)

            remaining_ops.append(int(rem_ops))
            remaining_work.append(int(rem_work))
            remaining_work_ratio.append(float(rem_work) / float(total_work))
            job_progress_ratio.append(
                float(total_ops - rem_ops) / float(total_ops)
            )
            post_route_tokens.append(self._post_route_tokens(job_id))

        current_cmax = self.get_makespan()
        total_remaining_work = int(sum(remaining_work))
        unfinished_jobs_count = sum(1 for x in remaining_ops if int(x) > 0)
        unfinished_jobs_ratio = (
            float(unfinished_jobs_count) / float(self.num_jobs) if self.num_jobs else 0.0
        )
        machine_ready_min = min(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_max = max(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_mean = (
            float(sum(self.machine_ready_time)) / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_var = (
            sum((float(x) - machine_ready_mean) ** 2 for x in self.machine_ready_time)
            / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_std = float(machine_ready_var ** 0.5)
        machine_remaining_load, machine_remaining_ops = self._remaining_machine_loads_and_ops()
        bottleneck_machine_load = max(machine_remaining_load) if machine_remaining_load else 0
        bottleneck_machine_id = (
            machine_remaining_load.index(bottleneck_machine_load)
            if machine_remaining_load
            else -1
        )
        bottleneck_machine_ops_left = (
            int(machine_remaining_ops[bottleneck_machine_id])
            if bottleneck_machine_id >= 0
            else 0
        )

        state = {
            "step_idx": self.scheduled_ops,
            "total_steps": self.total_ops,
            "scheduled_ratio": (
                float(self.scheduled_ops) / float(self.total_ops) if self.total_ops else 0.0
            ),
            "current_cmax": int(current_cmax),
            "job_next_op": list(self.job_next_op),
            "job_total_ops": list(self.job_total_ops),
            "job_total_work": list(self.job_total_work),
            "job_ready_time": list(self.job_ready_time),
            "machine_ready_time": list(self.machine_ready_time),
            "next_machine": next_machine,
            "next_proc_time": next_proc_time,
            "next2_machine": next2_machine,
            "next2_proc_time": next2_proc_time,
            "remaining_ops": remaining_ops,
            "remaining_work": remaining_work,
            "remaining_work_ratio": remaining_work_ratio,
            "job_progress_ratio": job_progress_ratio,
            "post_route_tokens": post_route_tokens,
            "total_remaining_work": int(total_remaining_work),
            "unfinished_jobs_count": int(unfinished_jobs_count),
            "unfinished_jobs_ratio": float(unfinished_jobs_ratio),
            "machine_ready_min": int(machine_ready_min),
            "machine_ready_mean": float(machine_ready_mean),
            "machine_ready_max": int(machine_ready_max),
            "machine_ready_std": float(machine_ready_std),
            "machine_remaining_load": machine_remaining_load,
            "machine_remaining_ops": machine_remaining_ops,
            "bottleneck_machine_id": int(bottleneck_machine_id),
            "bottleneck_machine_load": int(bottleneck_machine_load),
            "bottleneck_machine_ops_left": int(bottleneck_machine_ops_left),
            "feasible_jobs": self.get_feasible_jobs(),
        }
        return state

    def step(self, job_id: int) -> Tuple[Dict[str, object], float, bool, Dict[str, int]]:
        if self.is_done():
            raise ValueError("Cannot step: environment is already done.")
        if job_id < 0 or job_id >= self.num_jobs:
            raise ValueError(f"Invalid job_id: {job_id}.")

        op_idx = self.job_next_op[job_id]
        if op_idx >= self.operations_per_job[job_id]:
            raise ValueError(f"Job {job_id} has no remaining operation.")

        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        start_time = max(self.job_ready_time[job_id], self.machine_ready_time[machine_id])
        end_time = start_time + duration

        self.job_ready_time[job_id] = end_time
        self.machine_ready_time[machine_id] = end_time
        self.job_next_op[job_id] += 1
        self.scheduled_ops += 1

        event = {
            "step_idx": self.scheduled_ops - 1,
            "job_id": job_id,
            "op_idx": op_idx,
            "machine_id": machine_id,
            "duration": duration,
            "start_time": start_time,
            "end_time": end_time,
            "makespan_so_far": self.get_makespan(),
        }
        self.event_log.append(event)

        done = self.is_done()
        next_state = self.get_state_json()
        reward = 0.0
        return next_state, reward, done, event

    def rollout_teacher(self, job_sequence: Iterable[int]) -> List[Dict[str, object]]:
        """
        Roll out a full teacher sequence and collect step-level records.
        """
        self.reset()
        records: List[Dict[str, object]] = []
        for step_idx, job_id in enumerate(job_sequence):
            state_before = self.get_state_json()
            feasible_jobs = state_before["feasible_jobs"]
            if job_id not in feasible_jobs:
                raise ValueError(
                    f"Infeasible teacher action at step {step_idx}: "
                    f"job {job_id}, feasible={feasible_jobs}."
                )
            _, _, done, info = self.step(job_id)
            records.append(
                {
                    "step_idx": step_idx,
                    "state_json": state_before,
                    "feasible_jobs": list(feasible_jobs),
                    "target_job": job_id,
                    "info": info,
                    "done": done,
                }
            )
        if not self.is_done():
            raise ValueError(
                f"Teacher sequence ended early: scheduled {self.scheduled_ops}/{self.total_ops}."
            )
        return records

    def get_event_log(self) -> List[Dict[str, int]]:
        return list(self.event_log)

def _format_machine_ready(machine_ready_time: Sequence[int]) -> str:
    return " ".join(f"M{i}={t}" for i, t in enumerate(machine_ready_time))

def _format_feasible_jobs(feasible_jobs: Sequence[int]) -> str:
    if not feasible_jobs:
        return "[]"
    return "[" + ", ".join(f"Job {j}" for j in feasible_jobs) + "]"

def _format_action_codes(action_codes: Sequence[str]) -> str:
    if not action_codes:
        return "[]"
    return "[" + ", ".join(str(code) for code in action_codes) + "]"

def _format_route_tokens(route_tokens: Sequence[object]) -> str:
    if not route_tokens:
        return "[]"
    return "[" + ", ".join(str(token) for token in route_tokens) + "]"

def _route_contains_machine(route_tokens: Sequence[object], machine_id: int) -> bool:
    if machine_id < 0:
        return False
    prefix = f"M{machine_id}:"
    return any(str(token).startswith(prefix) for token in route_tokens)

def _effect_bottleneck_relation(
    effect: Dict[str, object],
    bottleneck_machine_id: int,
) -> str:
    machine_id = int(effect.get("machine_id", -1))
    remaining_ops_after = int(effect.get("remaining_ops_after", 0))
    post_route_tokens = effect.get("post_route_tokens", [])
    if bottleneck_machine_id < 0:
        return "unknown"
    if machine_id == bottleneck_machine_id:
        return "direct"
    if remaining_ops_after > 0 and _route_contains_machine(post_route_tokens, bottleneck_machine_id):
        return "releases_future_bottleneck"
    return "indirect"

def _effect_summary_lines(
    effect: Dict[str, object],
    state_json: Optional[Dict[str, object]] = None,
) -> List[str]:
    bottleneck_machine_id = (
        int(state_json.get("bottleneck_machine_id", -1))
        if state_json is not None
        else -1
    )
    return [
        f"- operation machine: {_machine_label(effect['machine_id'])}",
        f"- bottleneck_relation: {_effect_bottleneck_relation(effect, bottleneck_machine_id)}",
        f"- cmax: {int(effect['current_cmax_before'])}->{int(effect['current_cmax_after'])}",
        f"- delta_cmax: {int(effect['delta_cmax'])}",
        f"- est_start: {int(effect['estimated_start'])}",
        f"- est_end: {int(effect['estimated_end'])}",
        f"- machine_idle_gap: {int(effect['machine_idle_gap'])}",
        f"- job_wait: {int(effect['job_wait'])}",
        f"- remaining_ops_after: {int(effect['remaining_ops_after'])}",
        f"- remaining_work_after: {int(effect['remaining_work_after'])}",
        f"- machine_load: {int(effect['affected_machine_load'])}",
        f"- post_route: {_format_route_tokens(effect.get('post_route_tokens', []))}",
    ]

def build_reason_input_text(
    state_text: str,
    chosen_action_code: str,
    chosen_effect: Dict[str, object],
    action_effects: Sequence[Dict[str, object]],
    top_k: int = 3,
    state_json: Optional[Dict[str, object]] = None,
) -> str:
    alternatives = _sorted_reason_alternatives(
        action_effects=action_effects,
        chosen_action_code=chosen_action_code,
        top_k=top_k,
    )
    bottleneck_machine_id = (
        int(state_json.get("bottleneck_machine_id", -1))
        if state_json is not None
        else -1
    )
    bottleneck_load = (
        int(state_json.get("bottleneck_machine_load", 0))
        if state_json is not None
        else 0
    )
    bottleneck_ops_left = (
        int(state_json.get("bottleneck_machine_ops_left", 0))
        if state_json is not None
        else 0
    )
    lines = [
        "You are analyzing an already-selected JSSP action.",
        (
            "Objective: explain why this action was selected and why strong alternatives "
            "were not selected."
        ),
        (
            "Ground the explanation in explicit evidence only: immediate timing/Cmax impact, "
            "bottleneck-machine use or release, post-route exposure, "
            "waiting/idle trade-offs, and contrast against top alternatives."
        ),
        "",
        state_text,
        "",
        f"Selected action: {chosen_action_code}",
        "Critical context:",
        (
            f"- bottleneck_machine: {_machine_label(bottleneck_machine_id)} "
            f"(remaining_load={bottleneck_load}, ops_left={bottleneck_ops_left})"
        ),
        "Chosen transition:",
        *_effect_summary_lines(chosen_effect, state_json=state_json),
        "",
        "Top alternatives:",
    ]
    if alternatives:
        for alt in alternatives:
            lines.append(
                (
                    f"- {alt['action_code']}: "
                    f"operation machine={_machine_label(alt['machine_id'])}, "
                    f"post_route={_format_route_tokens(alt.get('post_route_tokens', []))}, "
                    f"cmax {int(alt['current_cmax_before'])}->{int(alt['current_cmax_after'])}, "
                    f"delta_cmax={int(alt['delta_cmax'])}, "
                    f"est_start={int(alt['estimated_start'])}, "
                    f"est_end={int(alt['estimated_end'])}, "
                    f"job_wait={int(alt['job_wait'])}, "
                    f"machine_idle_gap={int(alt['machine_idle_gap'])}, "
                    f"remaining_work_after={int(alt['remaining_work_after'])}, "
                    f"machine_load={int(alt['affected_machine_load'])}, "
                    f"bottleneck_relation={_effect_bottleneck_relation(alt, bottleneck_machine_id)}"
                )
            )
    else:
        lines.append("- (none)")

    lines.extend(
        [
            "",
            "Output format:",
            "Reason: ...",
            "Not chosen:",
            "- <Axxxx>: ...",
        ]
    )
    return "\n".join(lines)

def _chosen_bottleneck_clause(
    effect: Dict[str, object],
    bottleneck_machine_id: int,
    bottleneck_load: int,
    bottleneck_ops_left: int,
) -> str:
    machine_id = int(effect.get("machine_id", -1))
    affected_machine_load = int(effect.get("affected_machine_load", 0))
    post_route = _format_route_tokens(effect.get("post_route_tokens", []))
    if bottleneck_machine_id >= 0 and machine_id == bottleneck_machine_id:
        return (
            f"It directly activates the current bottleneck machine {_machine_label(machine_id)} "
            f"(remaining_load={bottleneck_load}, ops_left={bottleneck_ops_left}), so delaying this move "
            "would postpone work on the heaviest unresolved machine frontier."
        )
    if bottleneck_machine_id >= 0 and _route_contains_machine(effect.get("post_route_tokens", []), bottleneck_machine_id):
        return (
            f"Its post_route keeps future work on bottleneck "
            f"{_machine_label(bottleneck_machine_id)} visible and ready for earlier activation."
        )
    if bottleneck_machine_id >= 0 and affected_machine_load >= max(1, int(0.75 * bottleneck_load)):
        return (
            f"It works on high-load machine {_machine_label(machine_id)} "
            f"(remaining_load={affected_machine_load}), which is close to the current bottleneck pressure."
        )
    return (
        f"It advances the route on {_machine_label(machine_id)} without adding unnecessary "
        "timing friction to the current schedule state."
    )

def _chosen_progress_clause(effect: Dict[str, object]) -> str:
    rem_ops_before = int(effect.get("remaining_ops_before", 0))
    rem_ops_after = int(effect.get("remaining_ops_after", 0))
    rem_work_before = int(effect.get("remaining_work_before", 0))
    rem_work_after = int(effect.get("remaining_work_after", 0))
    if rem_ops_after <= 0:
        return (
            f"It completes this job, eliminating the remaining work from {rem_work_before} to 0."
        )
    post_route = _format_route_tokens(effect.get("post_route_tokens", []))
    if post_route != "[]":
        return (
            f"It reduces this job from remaining_work {rem_work_before}->{rem_work_after} "
            f"and remaining_ops {rem_ops_before}->{rem_ops_after}, leaving post_route={post_route}."
        )
    return (
        f"It reduces this job from remaining_work {rem_work_before}->{rem_work_after} "
        f"and remaining_ops {rem_ops_before}->{rem_ops_after}."
    )

def _chosen_wait_clause(effect: Dict[str, object]) -> Optional[str]:
    machine_idle_gap = int(effect.get("machine_idle_gap", 0))
    job_wait = int(effect.get("job_wait", 0))
    machine_id = int(effect.get("machine_id", -1))
    if machine_idle_gap > 0:
        return (
            f"It fills machine idle gap={machine_idle_gap} on {_machine_label(machine_id)}, "
            "which helps avoid leaving that machine unused."
        )
    if job_wait > 0:
        return (
            f"It intentionally waits {job_wait} time units for {_machine_label(machine_id)}, "
            "indicating that accessing this machine/route is worth the queue delay."
        )
    return "It can start immediately with no extra machine idle gap and no queue wait."

def _chosen_tradeoff_clause(
    chosen: Dict[str, object],
    best_alternative: Optional[Dict[str, object]],
) -> str:
    chosen_after = int(chosen.get("current_cmax_after", chosen.get("estimated_makespan_after", 0)))
    chosen_delta = int(chosen.get("delta_cmax", 0))
    if best_alternative is None:
        if chosen_delta == 0:
            return "It fits under the current Cmax and does not increase makespan immediately."
        return (
            f"It changes projected Cmax {int(chosen['current_cmax_before'])}->{chosen_after} "
            f"(delta={chosen_delta})."
        )

    best_after = int(
        best_alternative.get("current_cmax_after", best_alternative.get("estimated_makespan_after", 0))
    )
    gap = chosen_after - best_after
    if gap < 0:
        return (
            f"Among the strongest alternatives, it gives the smallest immediate projected Cmax "
            f"({chosen_after} vs {best_after})."
        )
    if gap == 0:
        if chosen_delta == 0:
            return "Its immediate projected Cmax matches the strongest alternatives while staying under the current makespan frontier."
        return (
            f"Its immediate projected Cmax matches the strongest alternative "
            f"({chosen_after}), so the tie is broken by route and bottleneck considerations."
        )
    if gap <= max(3, int(chosen.get("proc_time", 0)) // 2):
        return (
            f"It accepts only a small immediate Cmax penalty (+{gap} vs the strongest alternative) "
            "in exchange for better bottleneck/route progression."
        )
    return (
        f"Although its immediate projected Cmax is worse by +{gap} than the strongest alternative, "
        "the preference comes from stronger bottleneck alignment and downstream progression rather "
        "than short-term Cmax alone."
    )

def _alt_reason_line(
    alt: Dict[str, object],
    chosen: Dict[str, object],
    bottleneck_machine_id: int,
) -> str:
    alt_after = int(alt.get("current_cmax_after", alt.get("estimated_makespan_after", 0)))
    chosen_after = int(chosen.get("current_cmax_after", chosen.get("estimated_makespan_after", 0)))
    alt_machine = int(alt.get("machine_id", -1))
    chosen_machine = int(chosen.get("machine_id", -1))
    alt_post_route_tokens = alt.get("post_route_tokens", [])
    chosen_post_route_tokens = chosen.get("post_route_tokens", [])

    clauses: List[str] = []
    if alt_after > chosen_after:
        clauses.append(
            f"higher immediate projected Cmax ({alt_after} vs {chosen_after})"
        )
    elif alt_after < chosen_after:
        clauses.append(
            f"smaller immediate projected Cmax ({alt_after} vs {chosen_after}), but weaker structural progression"
        )
    else:
        clauses.append(
            f"same immediate projected Cmax ({alt_after})"
        )

    if bottleneck_machine_id >= 0 and chosen_machine == bottleneck_machine_id and alt_machine != bottleneck_machine_id:
        clauses.append(
            f"does not activate bottleneck {_machine_label(bottleneck_machine_id)}"
        )
    elif (
        bottleneck_machine_id >= 0
        and _route_contains_machine(chosen_post_route_tokens, bottleneck_machine_id)
        and not _route_contains_machine(alt_post_route_tokens, bottleneck_machine_id)
    ):
        clauses.append(
            f"does not pull the future bottleneck step on {_machine_label(bottleneck_machine_id)} forward"
        )
    elif int(alt.get("affected_machine_load", 0)) + 1 < int(chosen.get("affected_machine_load", 0)):
        clauses.append(
            f"works on a lighter machine (load={int(alt.get('affected_machine_load', 0))} vs {int(chosen.get('affected_machine_load', 0))})"
        )

    if int(alt.get("machine_idle_gap", 0)) > int(chosen.get("machine_idle_gap", 0)):
        clauses.append(
            f"larger machine idle gap ({int(alt.get('machine_idle_gap', 0))} vs {int(chosen.get('machine_idle_gap', 0))})"
        )
    elif int(alt.get("job_wait", 0)) > int(chosen.get("job_wait", 0)):
        clauses.append(
            f"more waiting before start ({int(alt.get('job_wait', 0))} vs {int(chosen.get('job_wait', 0))})"
        )

    rem_work_after = int(alt.get("remaining_work_after", 0))
    rem_ops_after = int(alt.get("remaining_ops_after", 0))
    post_route = _format_route_tokens(alt_post_route_tokens)
    if rem_ops_after > 0 and post_route != "[]":
        clauses.append(
            f"after this move it still leaves {rem_work_after} work and {rem_ops_after} ops, with post_route={post_route}"
        )
    else:
        clauses.append(
            f"after this move it still leaves {rem_work_after} work and {rem_ops_after} ops on that job"
        )

    clauses = clauses[:4]
    return "; ".join(clauses) + "."

def build_teacher_step_rationale(
    state_json: Dict[str, object],
    feasible_jobs: Sequence[int],
    chosen_job: int,
    action_code_to_job: Optional[Dict[str, int]] = None,
    max_not_chosen: int = 6,
    compute_action_effects_fn=None,
) -> str:
    """
    Build compact deterministic rationale text for supervision.

    Output format:
        Reason: ...
        Not chosen:
        - Job k: ...
    """
    feasible = [int(j) for j in feasible_jobs]
    chosen = int(chosen_job)
    if chosen not in feasible:
        feasible = feasible + [chosen]

    label_by_job: Dict[int, str]
    action_code_by_job: Dict[int, str]
    if action_code_to_job:
        label_by_job = invert_action_code_map(action_code_to_job)
        if chosen not in label_by_job:
            raise ValueError(
                f"chosen_job={chosen} is not found in action_code_to_job={action_code_to_job}"
            )
        action_code_by_job = dict(label_by_job)
    else:
        label_by_job = {j: f"Job {j}" for j in feasible}
        action_code_by_job = {j: f"Job {j}" for j in feasible}

    compute_fn = compute_action_effects_fn or compute_action_transition_features
    _, action_effects = compute_fn(
        state_json=state_json,
        action_code_to_job={action_code_by_job[j]: j for j in feasible},
    )
    feats = {int(effect["job_id"]): effect for effect in action_effects}
    c = feats[chosen]
    chosen_label = label_by_job[chosen]
    bottleneck_machine_id = int(state_json.get("bottleneck_machine_id", -1))
    bottleneck_load = int(state_json.get("bottleneck_machine_load", 0))
    bottleneck_ops_left = int(state_json.get("bottleneck_machine_ops_left", 0))

    best_alternative: Optional[Dict[str, object]] = None
    if len(feasible) > 1:
        alt_candidates = [feats[j] for j in feasible if j != chosen and j in feats]
        if alt_candidates:
            alt_candidates.sort(
                key=lambda f: (
                    int(f["estimated_makespan_after"]),
                    int(f["delta_cmax"]),
                    int(f["estimated_start"]),
                    int(f["proc_time"]),
                    int(f["job_id"]),
                )
            )
            best_alternative = alt_candidates[0]

    clauses = [
        (
            f"{chosen_label} is feasible on {_machine_label(c['machine_id'])} "
            f"(est_start={int(c['estimated_start'])}, est_end={int(c['estimated_end'])})."
        ),
        (
            f"Projected Cmax changes {int(c['current_cmax_before'])}->{int(c['current_cmax_after'])} "
            f"(delta={int(c['delta_cmax'])})."
        ),
        _chosen_progress_clause(c),
        _chosen_bottleneck_clause(
            c,
            bottleneck_machine_id=bottleneck_machine_id,
            bottleneck_load=bottleneck_load,
            bottleneck_ops_left=bottleneck_ops_left,
        ),
        _chosen_wait_clause(c),
        _chosen_tradeoff_clause(c, best_alternative),
    ]

    reason_line = "Reason: " + " ".join(clause for clause in clauses if clause)

    candidates = [j for j in feasible if j != chosen]
    candidates = [j for j in candidates if j in feats]
    candidates.sort(
        key=lambda j: (
            int(feats[j]["estimated_makespan_after"]),
            int(feats[j]["estimated_start"]),
            int(feats[j]["proc_time"]),
            j,
        )
    )
    limited = candidates[: max(0, int(max_not_chosen))]

    lines = [reason_line, "Not chosen:"]
    if not limited:
        lines.append("- (none)")
        return "\n".join(lines)

    for j in limited:
        f = feats[j]
        label = label_by_job[j]
        lines.append(f"- {label}: {_alt_reason_line(f, c, bottleneck_machine_id)}")
    return "\n".join(lines)

class DispatchJSSPStepEnv:
    """
    Event-driven JSSP environment.

    At each decision epoch, the agent chooses one dispatchable job.
    If more dispatchable jobs remain at the same current_time, the next decision
    happens without advancing time. If no dispatchable job remains, current_time
    jumps to the next operation completion event.
    """

    def __init__(self, inst_for_ortools: Sequence[Sequence[Sequence[int]]]):
        if not inst_for_ortools:
            raise ValueError("inst_for_ortools must not be empty.")

        self.inst_for_ortools: List[List[Tuple[int, int]]] = []
        for job_ops in inst_for_ortools:
            parsed_job_ops: List[Tuple[int, int]] = []
            for op in job_ops:
                if len(op) != 2:
                    raise ValueError(f"Each operation must be [machine, duration], got {op}.")
                machine, duration = int(op[0]), int(op[1])
                parsed_job_ops.append((machine, duration))
            self.inst_for_ortools.append(parsed_job_ops)

        self.num_jobs = len(self.inst_for_ortools)
        self.operations_per_job = [len(job_ops) for job_ops in self.inst_for_ortools]
        self.job_total_ops = list(self.operations_per_job)
        self.job_total_work = [
            sum(duration for _, duration in job_ops) for job_ops in self.inst_for_ortools
        ]
        self.total_ops = sum(self.operations_per_job)

        if self.total_ops <= 0:
            raise ValueError("total_ops must be positive.")

        max_machine_id = max(
            machine for job_ops in self.inst_for_ortools for machine, _ in job_ops
        )
        self.num_machines = max_machine_id + 1

        self.job_next_op: List[int] = []
        self.job_ready_time: List[int] = []
        self.machine_ready_time: List[int] = []
        self.current_time = 0
        self.scheduled_ops = 0
        self.running_ops: List[Dict[str, int]] = []
        self.event_log: List[Dict[str, int]] = []
        self.reset()

    @classmethod
    def from_prompt_jobs_first(
        cls, prompt_text: str, strict: bool = True
    ) -> "DispatchJSSPStepEnv":
        inst_for_ortools = parse_prompt_jobs_first(prompt_text, strict=strict)
        return cls(inst_for_ortools)

    def reset(self) -> Dict[str, object]:
        self.job_next_op = [0] * self.num_jobs
        self.job_ready_time = [0] * self.num_jobs
        self.machine_ready_time = [0] * self.num_machines
        self.current_time = 0
        self.scheduled_ops = 0
        self.running_ops = []
        self.event_log = []
        return self.get_state_json()

    def is_done(self) -> bool:
        return self.scheduled_ops == self.total_ops

    def get_makespan(self) -> int:
        return max(self.machine_ready_time) if self.machine_ready_time else 0

    def _next_operation(self, job_id: int, offset: int = 0) -> Tuple[int, int]:
        op_idx = self.job_next_op[job_id] + int(offset)
        if op_idx >= self.operations_per_job[job_id]:
            return -1, 0
        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        return int(machine_id), int(duration)

    def _remaining_work(self, job_id: int) -> int:
        next_op = self.job_next_op[job_id]
        return sum(duration for _, duration in self.inst_for_ortools[job_id][next_op:])

    def _post_route_tokens(self, job_id: int) -> List[str]:
        next_op = self.job_next_op[job_id] + 1
        return [
            f"M{int(machine_id)}:{int(duration)}"
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]
        ]

    def _remaining_machine_loads_and_ops(self) -> Tuple[List[int], List[int]]:
        machine_remaining_load = [0] * self.num_machines
        machine_remaining_ops = [0] * self.num_machines
        for job_id in range(self.num_jobs):
            next_op = self.job_next_op[job_id]
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]:
                machine_remaining_load[int(machine_id)] += int(duration)
                machine_remaining_ops[int(machine_id)] += 1
        return machine_remaining_load, machine_remaining_ops

    def _dispatchable(self, job_id: int) -> bool:
        if self.job_next_op[job_id] >= self.operations_per_job[job_id]:
            return False
        machine_id, duration = self._next_operation(job_id, offset=0)
        if machine_id < 0 or duration <= 0:
            return False
        return (
            int(self.job_ready_time[job_id]) <= int(self.current_time)
            and int(self.machine_ready_time[machine_id]) <= int(self.current_time)
        )

    def get_feasible_jobs(self) -> List[int]:
        return [job_id for job_id in range(self.num_jobs) if self._dispatchable(job_id)]

    def _advance_to_next_event(self) -> None:
        if not self.running_ops:
            return
        next_time = min(int(op["end_time"]) for op in self.running_ops)
        self.current_time = int(next_time)
        self.running_ops = [
            dict(op) for op in self.running_ops if int(op["end_time"]) > int(self.current_time)
        ]

    def _advance_until_decision_epoch(self) -> None:
        while not self.is_done() and not self.get_feasible_jobs():
            if not self.running_ops:
                raise RuntimeError(
                    "Dispatch environment deadlocked: no dispatchable jobs and no running ops."
                )
            self._advance_to_next_event()

    def get_state_json(self) -> Dict[str, object]:
        next_machine: List[int] = []
        next_proc_time: List[int] = []
        next2_machine: List[int] = []
        next2_proc_time: List[int] = []
        remaining_ops: List[int] = []
        remaining_work: List[int] = []
        remaining_work_ratio: List[float] = []
        job_progress_ratio: List[float] = []
        post_route_tokens: List[List[str]] = []

        for job_id in range(self.num_jobs):
            op_idx = self.job_next_op[job_id]
            machine, duration = self._next_operation(job_id, offset=0)
            machine2, duration2 = self._next_operation(job_id, offset=1)
            next_machine.append(machine)
            next_proc_time.append(duration)
            next2_machine.append(machine2)
            next2_proc_time.append(duration2)

            rem_ops = self.operations_per_job[job_id] - op_idx
            rem_work = self._remaining_work(job_id)
            total_work = max(int(self.job_total_work[job_id]), 1)
            total_ops = max(int(self.job_total_ops[job_id]), 1)

            remaining_ops.append(int(rem_ops))
            remaining_work.append(int(rem_work))
            remaining_work_ratio.append(float(rem_work) / float(total_work))
            job_progress_ratio.append(float(total_ops - rem_ops) / float(total_ops))
            post_route_tokens.append(self._post_route_tokens(job_id))

        current_cmax = self.get_makespan()
        total_remaining_work = int(sum(remaining_work))
        unfinished_jobs_count = sum(1 for x in remaining_ops if int(x) > 0)
        unfinished_jobs_ratio = (
            float(unfinished_jobs_count) / float(self.num_jobs) if self.num_jobs else 0.0
        )
        machine_ready_min = min(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_max = max(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_mean = (
            float(sum(self.machine_ready_time)) / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_var = (
            sum((float(x) - machine_ready_mean) ** 2 for x in self.machine_ready_time)
            / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_std = float(machine_ready_var ** 0.5)
        machine_remaining_load, machine_remaining_ops = self._remaining_machine_loads_and_ops()
        bottleneck_machine_load = max(machine_remaining_load) if machine_remaining_load else 0
        bottleneck_machine_id = (
            machine_remaining_load.index(bottleneck_machine_load)
            if machine_remaining_load
            else -1
        )
        bottleneck_machine_ops_left = (
            int(machine_remaining_ops[bottleneck_machine_id])
            if bottleneck_machine_id >= 0
            else 0
        )

        state = {
            "step_idx": self.scheduled_ops,
            "total_steps": self.total_ops,
            "scheduled_ratio": (
                float(self.scheduled_ops) / float(self.total_ops) if self.total_ops else 0.0
            ),
            "current_time": int(self.current_time),
            "current_cmax": int(current_cmax),
            "job_next_op": list(self.job_next_op),
            "job_total_ops": list(self.job_total_ops),
            "job_total_work": list(self.job_total_work),
            "job_ready_time": list(self.job_ready_time),
            "machine_ready_time": list(self.machine_ready_time),
            "next_machine": next_machine,
            "next_proc_time": next_proc_time,
            "next2_machine": next2_machine,
            "next2_proc_time": next2_proc_time,
            "remaining_ops": remaining_ops,
            "remaining_work": remaining_work,
            "remaining_work_ratio": remaining_work_ratio,
            "job_progress_ratio": job_progress_ratio,
            "post_route_tokens": post_route_tokens,
            "total_remaining_work": int(total_remaining_work),
            "unfinished_jobs_count": int(unfinished_jobs_count),
            "unfinished_jobs_ratio": float(unfinished_jobs_ratio),
            "machine_ready_min": int(machine_ready_min),
            "machine_ready_mean": float(machine_ready_mean),
            "machine_ready_max": int(machine_ready_max),
            "machine_ready_std": float(machine_ready_std),
            "machine_remaining_load": machine_remaining_load,
            "machine_remaining_ops": machine_remaining_ops,
            "bottleneck_machine_id": int(bottleneck_machine_id),
            "bottleneck_machine_load": int(bottleneck_machine_load),
            "bottleneck_machine_ops_left": int(bottleneck_machine_ops_left),
            "feasible_jobs": self.get_feasible_jobs(),
            "dispatchable_jobs": self.get_feasible_jobs(),
            "idle_machines": [
                int(machine_id)
                for machine_id, ready in enumerate(self.machine_ready_time)
                if int(ready) <= int(self.current_time)
            ],
            "running_operations": [dict(op) for op in sorted(
                self.running_ops,
                key=lambda x: (int(x["end_time"]), int(x["machine_id"]), int(x["job_id"])),
            )],
        }
        return state

    def step(self, job_id: int) -> Tuple[Dict[str, object], float, bool, Dict[str, int]]:
        if self.is_done():
            raise ValueError("Cannot step: environment is already done.")
        if job_id < 0 or job_id >= self.num_jobs:
            raise ValueError(f"Invalid job_id: {job_id}.")
        feasible_jobs = self.get_feasible_jobs()
        if job_id not in feasible_jobs:
            raise ValueError(
                f"Job {job_id} is not dispatchable at t={self.current_time}. feasible={feasible_jobs}"
            )

        op_idx = self.job_next_op[job_id]
        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        start_time = int(self.current_time)
        end_time = start_time + int(duration)

        self.job_ready_time[job_id] = end_time
        self.machine_ready_time[machine_id] = end_time
        self.job_next_op[job_id] += 1
        self.scheduled_ops += 1
        self.running_ops.append(
            {
                "job_id": int(job_id),
                "op_idx": int(op_idx),
                "machine_id": int(machine_id),
                "start_time": int(start_time),
                "end_time": int(end_time),
                "duration": int(duration),
            }
        )

        event = {
            "step_idx": self.scheduled_ops - 1,
            "job_id": int(job_id),
            "op_idx": int(op_idx),
            "machine_id": int(machine_id),
            "duration": int(duration),
            "start_time": int(start_time),
            "end_time": int(end_time),
            "decision_time": int(start_time),
            "makespan_so_far": self.get_makespan(),
        }
        self.event_log.append(event)

        done = self.is_done()
        if not done:
            self._advance_until_decision_epoch()
        next_state = self.get_state_json()
        reward = 0.0
        return next_state, reward, done, event

    def rollout_teacher(self, job_sequence: Iterable[int]) -> List[Dict[str, object]]:
        self.reset()
        records: List[Dict[str, object]] = []
        for step_idx, job_id in enumerate(job_sequence):
            state_before = self.get_state_json()
            feasible_jobs = state_before["feasible_jobs"]
            if job_id not in feasible_jobs:
                raise ValueError(
                    f"Infeasible teacher action at step {step_idx}: "
                    f"job {job_id}, feasible={feasible_jobs}."
                )
            _, _, done, info = self.step(job_id)
            records.append(
                {
                    "step_idx": step_idx,
                    "state_json": state_before,
                    "feasible_jobs": list(feasible_jobs),
                    "target_job": job_id,
                    "info": info,
                    "done": done,
                }
            )
        if not self.is_done():
            raise ValueError(
                f"Teacher sequence ended early: scheduled {self.scheduled_ops}/{self.total_ops}."
            )
        return records

    def get_event_log(self) -> List[Dict[str, int]]:
        return [dict(event) for event in self.event_log]

def _format_action_codes_dispatch(action_codes: Sequence[str]) -> str:
    if not action_codes:
        return "[]"
    return "[" + ", ".join(str(code) for code in action_codes) + "]"

def _format_value_dispatch(value: object) -> str:
    if isinstance(value, float):
        return f"{value:.4f}".rstrip("0").rstrip(".")
    return str(value)

def _machine_token_dispatch(machine_id: int) -> str:
    return f"M{int(machine_id)}" if int(machine_id) >= 0 else "M-"

def summarize_dispatch_state(state_json: Dict[str, object]) -> Dict[str, object]:
    machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
    current_cmax = int(
        state_json.get("current_cmax", max(machine_ready_time) if machine_ready_time else 0)
    )
    return {
        "step_idx": int(state_json["step_idx"]),
        "total_steps": int(state_json["total_steps"]),
        "scheduled_ratio": float(state_json.get("scheduled_ratio", 0.0)),
        "current_time": int(state_json.get("current_time", 0)),
        "current_cmax": current_cmax,
        "unfinished_jobs_count": int(state_json.get("unfinished_jobs_count", 0)),
        "unfinished_jobs_ratio": float(state_json.get("unfinished_jobs_ratio", 0.0)),
        "machine_ready_min": int(state_json.get("machine_ready_min", 0)),
        "machine_ready_mean": float(state_json.get("machine_ready_mean", 0.0)),
        "machine_ready_max": int(state_json.get("machine_ready_max", 0)),
        "machine_ready_std": float(state_json.get("machine_ready_std", 0.0)),
        "bottleneck_machine_id": int(state_json.get("bottleneck_machine_id", -1)),
        "bottleneck_machine_load": int(state_json.get("bottleneck_machine_load", 0)),
        "bottleneck_machine_ops_left": int(state_json.get("bottleneck_machine_ops_left", 0)),
    }

def compute_action_transition_features_dispatch(
    state_json: Dict[str, object],
    action_code_to_job: Dict[str, int],
) -> Tuple[int, List[Dict[str, object]]]:
    job_ready_time: List[int] = state_json["job_ready_time"]  # type: ignore[assignment]
    machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
    next_machine: List[int] = state_json["next_machine"]  # type: ignore[assignment]
    next_proc_time: List[int] = state_json["next_proc_time"]  # type: ignore[assignment]
    next2_machine: List[int] = state_json.get("next2_machine", [-1] * len(next_machine))  # type: ignore[assignment]
    next2_proc_time: List[int] = state_json.get("next2_proc_time", [0] * len(next_machine))  # type: ignore[assignment]
    remaining_ops: List[int] = state_json["remaining_ops"]  # type: ignore[assignment]
    remaining_work: List[int] = state_json["remaining_work"]  # type: ignore[assignment]
    job_total_ops: List[int] = state_json.get("job_total_ops", [1] * len(next_machine))  # type: ignore[assignment]
    job_total_work: List[int] = state_json.get("job_total_work", [1] * len(next_machine))  # type: ignore[assignment]
    machine_remaining_load: List[int] = state_json.get("machine_remaining_load", [0] * len(machine_ready_time))  # type: ignore[assignment]
    machine_remaining_ops: List[int] = state_json.get("machine_remaining_ops", [0] * len(machine_ready_time))  # type: ignore[assignment]
    post_route_tokens_by_job: List[List[str]] = state_json.get("post_route_tokens", [[] for _ in next_machine])  # type: ignore[assignment]

    current_time = int(state_json.get("current_time", 0))
    current_cmax = int(
        state_json.get("current_cmax", max(machine_ready_time) if machine_ready_time else 0)
    )
    total_remaining_work = int(
        state_json.get("total_remaining_work", sum(int(x) for x in remaining_work))
    )

    effects: List[Dict[str, object]] = []
    for action_code, job_id in action_code_to_job.items():
        job = int(job_id)
        machine_id = int(next_machine[job])
        proc_time = int(next_proc_time[job])
        if machine_id < 0 or proc_time <= 0:
            continue

        job_ready_before = int(job_ready_time[job])
        machine_ready_before = int(machine_ready_time[machine_id])
        est_start = max(current_time, job_ready_before, machine_ready_before)
        est_end = est_start + proc_time
        current_cmax_after = max(current_cmax, est_end)
        delta_cmax = current_cmax_after - current_cmax
        rem_ops_before = int(remaining_ops[job])
        rem_ops_after = max(0, rem_ops_before - 1)
        rem_work_before = int(remaining_work[job])
        rem_work_after = max(0, rem_work_before - proc_time)
        total_ops = max(int(job_total_ops[job]), 1)
        total_work = max(int(job_total_work[job]), 1)
        job_progress_ratio_before = float(total_ops - rem_ops_before) / float(total_ops)
        job_progress_ratio_after = float(total_ops - rem_ops_after) / float(total_ops)
        affected_machine_load = int(machine_remaining_load[machine_id])
        affected_machine_ops_left = int(machine_remaining_ops[machine_id])

        effects.append(
            {
                "action_code": str(action_code),
                "job_id": job,
                "machine_id": machine_id,
                "machine_token": _machine_token_dispatch(machine_id),
                "proc_time": proc_time,
                "next_machine": machine_id,
                "next_proc_time": proc_time,
                "next2_machine": int(next2_machine[job]),
                "next2_proc_time": int(next2_proc_time[job]),
                "remaining_ops_before": rem_ops_before,
                "remaining_ops_after": rem_ops_after,
                "remaining_work_before": rem_work_before,
                "remaining_work_after": rem_work_after,
                "job_progress_ratio_before": float(job_progress_ratio_before),
                "job_progress_ratio_after": float(job_progress_ratio_after),
                "job_ready_before": job_ready_before,
                "job_ready_after": int(est_end),
                "machine_ready_before": machine_ready_before,
                "machine_ready_after": int(est_end),
                "estimated_start": int(est_start),
                "estimated_end": int(est_end),
                "est_start": int(est_start),
                "est_end": int(est_end),
                "decision_time": int(current_time),
                "current_cmax_before": int(current_cmax),
                "current_cmax_after": int(current_cmax_after),
                "estimated_makespan_after": int(current_cmax_after),
                "delta_cmax": int(delta_cmax),
                "delta_cmax_ratio": (
                    float(delta_cmax) / float(max(current_cmax_after, 1))
                    if float(current_cmax_after) != 0.0
                    else 0.0
                ),
                "job_wait": int(max(0, current_time - job_ready_before)),
                "machine_idle_gap": int(max(0, current_time - machine_ready_before)),
                "slack_to_current_cmax": int(current_cmax - est_end),
                "affected_machine_load": affected_machine_load,
                "affected_machine_ops_left": affected_machine_ops_left,
                "affected_machine_load_ratio": (
                    float(affected_machine_load) / float(max(total_remaining_work, 1))
                ),
                "remaining_work_after_ratio": float(rem_work_after) / float(total_work),
                "post_route_tokens": list(post_route_tokens_by_job[job]),
                "post_route_len": int(len(post_route_tokens_by_job[job])),
            }
        )

    effects.sort(
        key=lambda x: (
            int(x["estimated_makespan_after"]),
            int(x["estimated_start"]),
            int(x["proc_time"]),
            int(x["job_id"]),
        )
    )
    return int(current_cmax), effects

def render_action_transition_line_dispatch(effect: Dict[str, object]) -> str:
    return (
        f"{effect['action_code']} | "
        f"operation machine={_machine_token(int(effect['next_machine']))} | "
        f"processing time={effect['next_proc_time']} | "
        f"decision_t={effect['decision_time']} | "
        f"est_start={effect['estimated_start']} | "
        f"est_end={effect['estimated_end']} | "
        f"cmax:{effect['current_cmax_before']}->{effect['current_cmax_after']} | "
        f"delta_cmax={effect['delta_cmax']} | "
        f"job_ready:{effect['job_ready_before']}->{effect['job_ready_after']} | "
        f"machine_ready:{effect['machine_ready_before']}->{effect['machine_ready_after']} | "
        f"rem_ops:{effect['remaining_ops_before']}->{effect['remaining_ops_after']} | "
        f"rem_work:{effect['remaining_work_before']}->{effect['remaining_work_after']} | "
        f"machine_load={effect['affected_machine_load']} | "
        f"machine_ops_left={effect['affected_machine_ops_left']} | "
        f"rem_work_after_ratio={_format_value(effect['remaining_work_after_ratio'])} | "
        f"post_route={_format_route_tokens(effect.get('post_route_tokens', []))}"
    )

def _order_prompt_effects_randomly_dispatch(
    state_json: Dict[str, object],
    action_effects: Sequence[Dict[str, object]],
) -> List[Dict[str, object]]:
    ordered_effects = [dict(effect) for effect in action_effects]
    seed_material = (
        f"{int(state_json.get('step_idx', 0))}|"
        f"{int(state_json.get('current_time', 0))}|"
        f"{int(state_json.get('current_cmax', 0))}|"
        + "|".join(sorted(str(effect.get("action_code", "")) for effect in ordered_effects))
    )
    rng = random.Random(seed_material)
    rng.shuffle(ordered_effects)
    return ordered_effects

def build_step_improvement_prompt(
    state_text: str,
    candidate_action_text: str,
    feasible_jobs: Sequence[object],
    reflection_memory: Optional[str] = None,
    step_diagnostics: Optional[str] = None,
) -> str:
    """Build a hindsight-aware step-improvement prompt."""
    feasible_str: str
    if feasible_jobs and isinstance(feasible_jobs[0], str):
        feasible_str = _format_action_codes([str(x) for x in feasible_jobs])
        output_format = "<Axxxx>"
    else:
        feasible_str = _format_feasible_jobs([int(x) for x in feasible_jobs])
        output_format = "Action: Job <id>"
    prompt = (
        "You are revising one decision inside a completed JSSP schedule.\n"
        "Use hindsight from the full episode, not only local greedy signals.\n"
        "If a small short-term sacrifice helps earlier bottleneck activation, lower idle loss, or better downstream route progression, prefer it.\n\n"
        "Current step state:\n"
        f"{state_text}\n\n"
        "Previously chosen action in the failed/improvable rollout:\n"
        f"{candidate_action_text}\n\n"
        "Rules:\n"
        "- Objective: minimize final makespan (Cmax).\n"
        "- Think in hindsight: ask which choice would have reduced downstream bottleneck delay, idle gaps, or route blocking.\n"
        "- Prefer the action that best aligns with bottleneck-machine usage, downstream route progression, and lower regret against strong alternatives.\n"
        f"- Final action must be one of feasible options: {feasible_str}\n"
        f"- Return exactly one output in this format: {output_format}\n"
        "- Do not output explanation.\n"
        "- Do not repeat the previous action if the reflection evidence says an alternative is structurally better.\n"
    )
    if reflection_memory:
        prompt += f"\nEpisode hindsight reflection:\n{reflection_memory}\n"
    if step_diagnostics:
        prompt += f"\nCritical-step diagnostics:\n{step_diagnostics}\n"
    return prompt


def build_step_prompt_dispatch(
    state_json: Dict[str, object],
    feasible_jobs: Sequence[int],
    step_idx: int,
    total_steps: int,
    problem_context_text: Optional[str] = None,
    action_code_to_job: Optional[Dict[str, int]] = None,
) -> str:
    lines = [
        "You are solving JSSP with event-driven dispatching.",
        "Objective: minimize final makespan (Cmax) while respecting precedence and machine availability.",
    ]
    if problem_context_text:
        lines.extend(["Static problem context:", problem_context_text])

    summary = summarize_dispatch_state(state_json)
    idle_machines = [int(x) for x in state_json.get("idle_machines", [])]  # type: ignore[arg-type]
    running_operations = list(state_json.get("running_operations", []) or [])
    lines.extend(
        [
            "Dispatch state:",
            (
                f"decision_step={int(summary['step_idx']) + 1}/{int(summary['total_steps'])} "
                f"scheduled_ratio={_format_value_dispatch(summary['scheduled_ratio'])}"
            ),
            f"current_time={summary['current_time']}",
            f"current_cmax={summary['current_cmax']}",
            (
                f"unfinished_jobs_count={summary['unfinished_jobs_count']} "
                f"unfinished_jobs_ratio={_format_value_dispatch(summary['unfinished_jobs_ratio'])}"
            ),
            (
                f"idle_machines={[ _machine_token_dispatch(m) for m in idle_machines ]} "
                f"num_running_ops={len(running_operations)}"
            ),
            (
                f"machine_ready_min={summary['machine_ready_min']} "
                f"machine_ready_mean={_format_value_dispatch(summary['machine_ready_mean'])} "
                f"machine_ready_max={summary['machine_ready_max']} "
                f"machine_ready_std={_format_value_dispatch(summary['machine_ready_std'])}"
            ),
            (
                f"bottleneck_machine={_machine_token_dispatch(int(summary['bottleneck_machine_id']))} "
                f"bottleneck_load={summary['bottleneck_machine_load']} "
                f"bottleneck_ops_left={summary['bottleneck_machine_ops_left']}"
            ),
        ]
    )
    if running_operations:
        lines.append("Running operations:")
        for op in running_operations:
            lines.append(
                f"Job {int(op['job_id'])} Op {int(op['op_idx'])} on "
                f"{_machine_token_dispatch(int(op['machine_id']))}: "
                f"{int(op['start_time'])}->{int(op['end_time'])}"
            )

    if action_code_to_job:
        action_codes = list(action_code_to_job.keys())
        _, action_effects = compute_action_transition_features_dispatch(
            state_json=state_json,
            action_code_to_job=action_code_to_job,
        )
        lines.extend(
            [
                f"Dispatchable action codes now: {_format_action_codes_dispatch(action_codes)}",
                "Candidate dispatch effects:",
            ]
        )
        prompt_effects = _order_prompt_effects_randomly_dispatch(state_json, action_effects)
        for effect in prompt_effects:
            lines.append(render_action_transition_line_dispatch(effect))
        lines.extend(
            [
                "Action codes are randomized at each decision epoch. Do not assume persistent code identity.",
                "Choose exactly one dispatchable action code.",
                "Return exactly one code from the dispatchable action set, for example <A3812>.",
            ]
        )
    else:
        lines.extend(
            [
                f"Dispatchable jobs: {list(int(j) for j in feasible_jobs)}",
                "Choose exactly one dispatchable job.",
                "Return exactly one line: Action: Job <id>",
            ]
        )
    return "\n".join(lines)

@dataclass(frozen=True)
class _StepStack:
    env_cls: type
    build_problem_context_text: callable
    build_randomized_action_code_map: callable
    build_step_prompt: callable
    build_step_improvement_prompt: callable
    compute_action_transition_features: callable
    invert_action_code_map: callable


def resolve_step_stack(env_mode: str = 'serial') -> _StepStack:
    normalized = str(env_mode or 'serial').strip().lower()
    if normalized == 'serial':
        return _StepStack(
            env_cls=StaticJSSPStepEnv,
            build_problem_context_text=build_problem_context_text,
            build_randomized_action_code_map=build_randomized_action_code_map,
            build_step_prompt=build_step_prompt,
            build_step_improvement_prompt=build_step_improvement_prompt,
            compute_action_transition_features=compute_action_transition_features,
            invert_action_code_map=invert_action_code_map,
        )
    if normalized == 'dispatch':
        return _StepStack(
            env_cls=DispatchJSSPStepEnv,
            build_problem_context_text=build_problem_context_text,
            build_randomized_action_code_map=build_randomized_action_code_map,
            build_step_prompt=build_step_prompt_dispatch,
            build_step_improvement_prompt=build_step_improvement_prompt,
            compute_action_transition_features=compute_action_transition_features_dispatch,
            invert_action_code_map=invert_action_code_map,
        )
    raise ValueError(f"Unsupported env_mode={env_mode}. Use 'serial' or 'dispatch'.")

def _build_step_chat_prompt(tokenizer, state_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert JSSP scheduler. "
                "Primary objective: minimize final makespan (Cmax). "
                "Choose exactly one feasible action code for the current step. "
                "Output exactly one code such as <A3812> and nothing else."
            ),
        },
        {
            "role": "user",
            "content": state_text,
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def _build_step_improvement_chat_prompt(tokenizer, improvement_prompt_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are refining one JSSP step action. "
                "Primary objective: minimize final makespan (Cmax). "
                "Return exactly one feasible action code such as <A3812>."
            ),
        },
        {
            "role": "user",
            "content": improvement_prompt_text,
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def _build_step_rationale_chat_prompt(tokenizer, rationale_prompt_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You explain a fixed scheduling decision. "
                "Focus on makespan minimization (Cmax). "
                "Do not output a new action. Follow the requested explanation format only."
            ),
        },
        {
            "role": "user",
            "content": rationale_prompt_text,
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def _activate_adapter_if_available(model, adapter_name: str):
    if hasattr(model, "set_adapter"):
        model.set_adapter(adapter_name)

def _maybe_load_reason_adapter(model, reason_model_path: str | None):
    if not reason_model_path:
        return False
    if not hasattr(model, "load_adapter"):
        raise RuntimeError(
            "Reason adapter path was provided, but the loaded model does not support load_adapter()."
        )
    model.load_adapter(reason_model_path, adapter_name="reason")
    _activate_adapter_if_available(model, "default")
    return True

def _normalize_env_mode(env_mode: str | None) -> str:
    return str(env_mode or "serial").strip().lower()

def _coerce_run_args(run_args=None, cfg=None):
    resolved = run_args if run_args is not None else cfg
    if resolved is None:
        raise ValueError("Either run_args or cfg must be provided.")
    if isinstance(resolved, dict):
        return SimpleNamespace(**resolved)
    return resolved

def _sample_step_action(
    model,
    tokenizer,
    prompt_text: str,
    feasible_action_codes,
    use_masking: bool = True,
    max_new_tokens: int = 8,
    action_code_width: int = 4,
    action_code_cap: int = 9999,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer([prompt_text], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    prompt_len = inputs["input_ids"].shape[1]
    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": True,
        "temperature": 1.0,
        "top_k": 50,
        "top_p": 0.95,
        "num_return_sequences": 1,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if use_masking:
        generation_kwargs["prefix_allowed_tokens_fn"] = build_step_prefix_allowed_tokens_fn(
            tokenizer=tokenizer,
            feasible_action_codes_provider=list(feasible_action_codes),
            prompt_len=prompt_len,
            code_width=action_code_width,
            code_cap=action_code_cap,
        )

    outputs = model.generate(**inputs, **generation_kwargs)
    generated_ids = outputs[0][prompt_len:]

    if not feasible_action_codes:
        raise StepActionParseError("No feasible action codes available at this step.")
    if generated_ids.numel() <= 0:
        raise StepActionParseError("No action code was generated.")

    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=False).strip()
    chosen_action_code = parse_action_code(generated_text, code_width=action_code_width)
    if chosen_action_code is None:
        raise StepActionParseError(
            f"Failed to parse generated text into action code. output={generated_text!r}"
        )

    if str(chosen_action_code) not in feasible_action_codes:
        raise StepActionParseError(
            "Parsed action code is not feasible. "
            f"parsed={chosen_action_code}, feasible={list(feasible_action_codes)}, output={generated_text!r}"
        )

    return str(chosen_action_code), generated_text

def _sample_step_rationale(
    model,
    tokenizer,
    prompt_text: str,
    max_new_tokens: int = 96,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer([prompt_text], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_k=40,
        top_p=0.9,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][prompt_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return generated_text

def _deterministic_step_rationale(
    chosen_action_code: str,
    chosen_effect: dict,
    action_effects: list,
) -> str:
    chosen_ms = int(chosen_effect["estimated_makespan_after"])
    best_alt = None
    for opt in action_effects:
        if str(opt.get("action_code")) == str(chosen_action_code):
            continue
        ms = int(opt.get("estimated_makespan_after", chosen_ms))
        if best_alt is None or ms < best_alt:
            best_alt = ms
    if best_alt is None:
        compare_text = "no alternative feasible option at this step."
    else:
        gap = int(chosen_ms - best_alt)
        if gap <= 0:
            compare_text = "best estimated Cmax among feasible options."
        else:
            compare_text = f"estimated Cmax is +{gap} versus the best alternative."
    return (
        f"{chosen_action_code}: feasible on M{int(chosen_effect['machine_id'])} "
        f"(est_start={int(chosen_effect['estimated_start'])}, est_end={int(chosen_effect['estimated_end'])}, "
        f"proc={int(chosen_effect['proc_time'])}), "
        f"Cmax {int(chosen_effect['current_cmax_before'])}->{int(chosen_effect['current_cmax_after'])} "
        f"(delta={int(chosen_effect['delta_cmax'])}), "
        f"machine_idle_gap={int(chosen_effect['machine_idle_gap'])}; {compare_text}"
    )

def _clean_step_rationale(
    raw_text: str,
    chosen_action_code: str,
    chosen_effect: dict,
    action_effects: list,
) -> str:
    text = (raw_text or "").strip()
    if not text:
        return _deterministic_step_rationale(
            chosen_action_code, chosen_effect, action_effects
        )

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    lines = [ln for ln in lines if not ln.lower().startswith("action:")]

    reason = None
    for ln in lines:
        if ln.lower().startswith("reason:"):
            reason = ln.split(":", 1)[1].strip()
            break

    if reason is None:
        for ln in lines:
            low = ln.lower()
            if low.startswith("not chosen"):
                continue
            if low.startswith("- "):
                continue
            reason = ln
            break

    if not reason:
        return _deterministic_step_rationale(
            chosen_action_code, chosen_effect, action_effects
        )

    if "Not chosen:" in reason:
        reason = reason.split("Not chosen:", 1)[0].strip()

    found_codes = re.findall(r"<\s*[sS]\s*\d+\s*>", reason)
    normalized_codes = {re.sub(r"\s+", "", code).upper() for code in found_codes}
    chosen_norm = re.sub(r"\s+", "", str(chosen_action_code)).upper()
    if any(code != chosen_norm for code in normalized_codes):
        return _deterministic_step_rationale(
            chosen_action_code, chosen_effect, action_effects
        )

    reason = re.sub(r"<\s*[sS]\s*\d+\s*>", str(chosen_action_code), reason)
    if len(reason) > 220:
        reason = reason[:217].rstrip() + "..."
    return f"{chosen_action_code}: {reason}"

def _estimate_action_effects(state_json, action_code_to_job, env_mode: str = "serial"):
    step_stack = resolve_step_stack(_normalize_env_mode(env_mode))
    return step_stack.compute_action_transition_features(state_json, action_code_to_job)

def _print_step_trace(raw_step_outputs):
    for step in raw_step_outputs:
        chosen = (
            f"{step['chosen_action_code']} -> Job {step['chosen_job']} "
            f"(M{step['machine_id']}, t={step['chosen_proc_time']}, "
            f"start={step['chosen_start_time']}, end={step['chosen_end_time']})"
        )
        print(
            f"[Step {int(step['step_idx']):03d}] "
            f"makespan {int(step['makespan_before'])} -> {int(step['makespan_after'])} | chosen: {chosen}"
        )
        rationale = step.get("rationale_text")
        if rationale:
            print(f"  reason: {rationale}")
        not_chosen = step.get("not_chosen_options", [])
        if not_chosen:
            parts = []
            for opt in not_chosen:
                parts.append(
                    f"{opt['action_code']}->Job{opt['job_id']}"
                    f"(M{opt['machine_id']},t={opt['proc_time']},ms={opt['estimated_makespan_after']},"
                    f"dC={opt['delta_cmax']})"
                )
            print("  not_chosen:", "; ".join(parts))

def _best_alternative_option(step):
    chosen_code = str(step.get("chosen_action_code"))
    alternatives = [
        opt
        for opt in step.get("all_options", [])
        if str(opt.get("action_code")) != chosen_code
    ]
    if not alternatives:
        return None
    return min(
        alternatives,
        key=lambda x: (
            int(x.get("estimated_makespan_after", 10**12)),
            int(x.get("estimated_start", 10**12)),
            int(x.get("proc_time", 10**12)),
        ),
    )

def _critical_step_score(step):
    best_alt = _best_alternative_option(step)
    if best_alt is None:
        return None
    chosen_ms = int(step.get("chosen_estimated_makespan_after", step.get("makespan_after", 0)))
    best_alt_ms = int(best_alt.get("estimated_makespan_after", chosen_ms))
    immediate_gap = int(chosen_ms - best_alt_ms)
    makespan_jump = int(step.get("makespan_after", 0) - step.get("makespan_before", 0))
    chosen_start = int(step.get("chosen_start_time", 0))
    best_alt_start = int(best_alt.get("estimated_start", chosen_start))
    start_delay = int(chosen_start - best_alt_start)
    if immediate_gap <= 0 and makespan_jump <= 0 and start_delay <= 0:
        return None
    return (immediate_gap, makespan_jump, start_delay, int(step.get("step_idx", -1)))

def _select_top_critical_steps(raw_step_outputs, top_k: int = 3):
    scored = []
    for step in raw_step_outputs:
        score = _critical_step_score(step)
        if score is None:
            continue
        scored.append((score, step))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [step for _, step in scored[: max(1, int(top_k))]]

def _select_critical_step(raw_step_outputs):
    top = _select_top_critical_steps(raw_step_outputs, top_k=1)
    return top[0] if top else None

def _build_step_diagnostics(step):
    chosen_code = str(step.get("chosen_action_code"))
    chosen_job = int(step.get("chosen_job", -1))
    chosen_ms = int(step.get("chosen_estimated_makespan_after", step.get("makespan_after", 0)))
    chosen_start = int(step.get("chosen_start_time", 0))
    chosen_machine = int(step.get("machine_id", -1))
    chosen_end = int(step.get("chosen_end_time", step.get("makespan_after", 0)))
    chosen_effect = None
    alternatives = [
        opt
        for opt in step.get("all_options", [])
        if str(opt.get("action_code")) != chosen_code
    ]
    for opt in step.get("all_options", []):
        if str(opt.get("action_code")) == chosen_code:
            chosen_effect = opt
            break
    alternatives = sorted(
        alternatives,
        key=lambda x: (
            int(x.get("estimated_makespan_after", 10**12)),
            int(x.get("estimated_start", 10**12)),
        ),
    )
    lines = [
        (
            f"Chosen={chosen_code}/Job{chosen_job}, machine=M{chosen_machine}, "
            f"start={chosen_start}, end={chosen_end}, est_Cmax_after={chosen_ms}"
        )
    ]
    if chosen_effect is not None:
        chosen_post_route = _format_route_tokens(chosen_effect.get("post_route_tokens", []))
        lines.append(
            (
                "Chosen details: "
                f"delta_cmax={int(chosen_effect.get('delta_cmax', 0))}, "
                f"job_wait={int(chosen_effect.get('job_wait', 0))}, "
                f"machine_idle_gap={int(chosen_effect.get('machine_idle_gap', 0))}, "
                f"post_route={chosen_post_route}, "
                f"machine_load={int(chosen_effect.get('affected_machine_load', 0))}"
            )
        )
    for rank, opt in enumerate(alternatives[:3], start=1):
        alt_post_route = _format_route_tokens(opt.get("post_route_tokens", []))
        lines.append(
            (
                f"Alt{rank}={opt['action_code']}/Job{int(opt['job_id'])}, "
                f"est_Cmax_after={int(opt['estimated_makespan_after'])}, "
                f"est_start={int(opt['estimated_start'])}, "
                f"proc={int(opt['proc_time'])}, "
                f"delta_cmax={int(opt.get('delta_cmax', 0))}, "
                f"idle_gap={int(opt.get('machine_idle_gap', 0))}, "
                f"job_wait={int(opt.get('job_wait', 0))}, "
                f"post_route={alt_post_route}, "
                f"machine_load={int(opt.get('affected_machine_load', 0))}"
            )
        )
    return "\n".join(lines)

def _build_reflection_memory(current_makespan: int, critical_steps):
    lines = [f"Current episode makespan={int(current_makespan)}."]
    if not critical_steps:
        lines.append("No critical-step hindsight available.")
        return "\n".join(lines)

    increase_count = sum(
        1 for step in critical_steps
        if int(step.get("makespan_after", 0)) > int(step.get("makespan_before", 0))
    )
    bottleneck_machine_counts = {}
    for step in critical_steps:
        for opt in step.get("all_options", []):
            machine_id = int(opt.get("machine_id", -1))
            bottleneck_machine_counts[machine_id] = bottleneck_machine_counts.get(machine_id, 0) + 1
    dominant_machine = None
    if bottleneck_machine_counts:
        dominant_machine = max(
            bottleneck_machine_counts.items(),
            key=lambda x: (x[1], -x[0]),
        )[0]

    lines.extend(
        [
            (
                "Episode postmortem: the schedule likely lost quality through one or more of the "
                "following patterns: delaying bottleneck activation, accepting larger idle/wait gaps "
                "without structural payoff, or choosing weaker post-routes when immediate Cmax "
                "signals were close."
            ),
            (
                f"Critical-step count={len(critical_steps)}, "
                f"makespan-increase steps among them={increase_count}."
            ),
        ]
    )
    if dominant_machine is not None and dominant_machine >= 0:
        lines.append(
            f"Dominant machine appearing in critical alternatives: M{dominant_machine}. Treat this as a likely bottleneck focus."
        )
    lines.extend(
        [
            "Reflection rules:",
            "1. Prefer actions that activate or release the bottleneck route earlier.",
            "2. If immediate projected Cmax is tied or close, prefer lower regret in post-route progression.",
            "3. Avoid larger machine idle gaps or waits unless they clearly unlock stronger bottleneck progress.",
            "4. Do not repeat previously identified bad choices when a strong feasible alternative exists.",
        ]
    )
    for rank, step in enumerate(critical_steps, start=1):
        best_alt = _best_alternative_option(step)
        if best_alt is None:
            continue
        chosen_code = str(step.get("chosen_action_code"))
        chosen_job = int(step.get("chosen_job", -1))
        chosen_ms = int(step.get("chosen_estimated_makespan_after", step.get("makespan_after", 0)))
        chosen_start = int(step.get("chosen_start_time", 0))
        alt_code = str(best_alt.get("action_code"))
        alt_job = int(best_alt.get("job_id", -1))
        alt_ms = int(best_alt.get("estimated_makespan_after", chosen_ms))
        alt_start = int(best_alt.get("estimated_start", chosen_start))
        alt_delta = int(best_alt.get("delta_cmax", 0))
        alt_idle_gap = int(best_alt.get("machine_idle_gap", 0))
        alt_job_wait = int(best_alt.get("job_wait", 0))
        alt_post_route = _format_route_tokens(best_alt.get("post_route_tokens", []))
        chosen_idle_gap = int(
            next(
                (
                    opt.get("machine_idle_gap", 0)
                    for opt in step.get("all_options", [])
                    if str(opt.get("action_code")) == chosen_code
                ),
                0,
            )
        )
        chosen_job_wait = int(
            next(
                (
                    opt.get("job_wait", 0)
                    for opt in step.get("all_options", [])
                    if str(opt.get("action_code")) == chosen_code
                ),
                0,
            )
        )
        lines.append(
            (
                f"Rule {rank}: step {int(step.get('step_idx', -1))}. "
                f"Previous choice {chosen_code}/Job{chosen_job} had est_Cmax={chosen_ms}, "
                f"start={chosen_start}, idle_gap={chosen_idle_gap}, job_wait={chosen_job_wait}. "
                f"Prefer {alt_code}/Job{alt_job} instead when feasible because it gives "
                f"est_Cmax={alt_ms}, start={alt_start}, delta_cmax={alt_delta}, "
                f"idle_gap={alt_idle_gap}, job_wait={alt_job_wait}, and post_route={alt_post_route}."
            )
        )
    return "\n".join(lines)

def _run_single_step_rollout(
    model,
    tokenizer,
    inst_for_ortools,
    env_mode: str = "serial",
    use_masking: bool = True,
    step_action_max_new_tokens: int = 8,
    include_problem_context: bool = True,
    enable_step_improvement: bool = False,
    step_reflection_passes: int = 1,
    emit_step_rationale: bool = False,
    step_rationale_max_new_tokens: int = 96,
    reason_topk: int = 3,
    use_reason_adapter: bool = False,
    action_code_width: int = 4,
    action_code_seed: int = 42,
    action_code_cap: int = 9999,
    guidance_by_step=None,
    reflection_memory_text: str | None = None,
    replay_action_jobs_by_step=None,
    replay_prefix_until: int | None = None,
    print_step_trace: bool = False,
):
    step_stack = resolve_step_stack(_normalize_env_mode(env_mode))
    env = step_stack.env_cls(inst_for_ortools)
    env.reset()
    raw_step_outputs = []
    problem_context_text = (
        step_stack.build_problem_context_text(inst_for_ortools)
        if include_problem_context
        else None
    )
    action_rng = random.Random(int(action_code_seed))
    guidance_map = guidance_by_step or {}

    while not env.is_done():
        state = env.get_state_json()
        feasible_jobs = list(state["feasible_jobs"])
        step_idx = int(state["step_idx"])
        action_code_to_job = step_stack.build_randomized_action_code_map(
            feasible_jobs=feasible_jobs,
            rng=action_rng,
            code_width=action_code_width,
            code_start=1,
            code_cap=action_code_cap,
        )
        makespan_before, action_effects = _estimate_action_effects(
            state_json=state,
            action_code_to_job=action_code_to_job,
            env_mode=env_mode,
        )
        effect_by_code = {x["action_code"]: x for x in action_effects}
        job_to_action_code = step_stack.invert_action_code_map(action_code_to_job)
        feasible_action_codes = list(action_code_to_job.keys())
        prompt_state_text = step_stack.build_step_prompt(
            state_json=state,
            feasible_jobs=feasible_jobs,
            step_idx=step_idx,
            total_steps=state["total_steps"],
            problem_context_text=problem_context_text,
            action_code_to_job=action_code_to_job,
        )
        if reflection_memory_text:
            prompt_state_text = (
                f"{prompt_state_text}\n"
                f"Episode reflection memory:\n{reflection_memory_text}\n"
                "Apply these reflection rules while selecting this step action."
            )
        if step_idx in guidance_map:
            step_guidance = dict(guidance_map[step_idx])
            preferred_job = int(step_guidance.get("preferred_job", -1))
            avoid_jobs = [
                int(x) for x in step_guidance.get("avoid_jobs", [])
                if int(x) >= 0
            ]
            reason_text = str(step_guidance.get("reason", "")).strip()
            guide_lines = [
                "Post-episode guidance: This step was identified as a Cmax bottleneck.",
            ]
            if preferred_job >= 0:
                guide_lines.append(f"If feasible, prefer Job {preferred_job}.")
            if avoid_jobs:
                guide_lines.append(
                    "Avoid these jobs if strong alternatives exist: "
                    + ", ".join(f"Job {j}" for j in avoid_jobs)
                )
            if reason_text:
                guide_lines.append(f"Why: {reason_text}")
            prompt_state_text = f"{prompt_state_text}\n" + "\n".join(guide_lines)
        is_replay_prefix = (
            replay_action_jobs_by_step is not None
            and replay_prefix_until is not None
            and int(step_idx) < int(replay_prefix_until)
        )
        if is_replay_prefix:
            chosen_job = int(replay_action_jobs_by_step[int(step_idx)])
            if chosen_job not in feasible_jobs:
                raise RuntimeError(
                    "Prefix replay became infeasible before the guided step. "
                    f"step_idx={step_idx}, replay_job={chosen_job}, feasible={feasible_jobs}"
                )
            chosen_action_code = str(job_to_action_code[int(chosen_job)])
            initial_model_text = f"[REPLAY] {chosen_action_code}"
        else:
            prompt_text = _build_step_chat_prompt(tokenizer, prompt_state_text)
            if use_reason_adapter:
                _activate_adapter_if_available(model, "default")
            chosen_action_code, initial_model_text = _sample_step_action(
                model=model,
                tokenizer=tokenizer,
                prompt_text=prompt_text,
                feasible_action_codes=feasible_action_codes,
                use_masking=use_masking,
                max_new_tokens=step_action_max_new_tokens,
                action_code_width=action_code_width,
                action_code_cap=action_code_cap,
            )
            chosen_job = int(action_code_to_job[chosen_action_code])
        chosen_effect = effect_by_code.get(chosen_action_code)
        if chosen_effect is None:
            raise RuntimeError(
                "Internal mismatch: chosen action code is missing in step option effects. "
                f"chosen_action_code={chosen_action_code}, available={list(effect_by_code.keys())}"
            )

        rationale_text = None
        if emit_step_rationale:
            if is_replay_prefix:
                rationale_text = _deterministic_step_rationale(
                    chosen_action_code=chosen_action_code,
                    chosen_effect=chosen_effect,
                    action_effects=action_effects,
                )
            else:
                rationale_prompt = build_reason_input_text(
                    state_text=prompt_state_text,
                    chosen_action_code=chosen_action_code,
                    chosen_effect=chosen_effect,
                    action_effects=action_effects,
                    top_k=int(reason_topk),
                    state_json=state,
                )
                rationale_chat_prompt = _build_step_rationale_chat_prompt(
                    tokenizer, rationale_prompt
                )
                try:
                    if use_reason_adapter:
                        _activate_adapter_if_available(model, "reason")
                    rationale_raw = _sample_step_rationale(
                        model=model,
                        tokenizer=tokenizer,
                        prompt_text=rationale_chat_prompt,
                        max_new_tokens=step_rationale_max_new_tokens,
                    )
                finally:
                    if use_reason_adapter:
                        _activate_adapter_if_available(model, "default")
                rationale_text = _clean_step_rationale(
                    rationale_raw,
                    chosen_action_code=chosen_action_code,
                    chosen_effect=chosen_effect,
                    action_effects=action_effects,
                )

        _, _, _, info = env.step(chosen_job)
        not_chosen = [
            x for x in action_effects
            if str(x["action_code"]) != str(chosen_action_code)
        ]
        step_record = {
            "step_idx": int(info["step_idx"]),
            "feasible_jobs": feasible_jobs,
            "feasible_action_codes": feasible_action_codes,
            "action_code_to_job": action_code_to_job,
            "state_text": prompt_state_text,
            "model_output": initial_model_text,
            "rationale_text": rationale_text,
            "chosen_action_code": chosen_action_code,
            "chosen_job": int(chosen_job),
            "op_idx": int(info["op_idx"]),
            "machine_id": int(info["machine_id"]),
            "chosen_start_time": int(info["start_time"]),
            "chosen_end_time": int(info["end_time"]),
            "chosen_proc_time": int(info.get("duration", chosen_effect["proc_time"])),
            "makespan_before": int(makespan_before),
            "makespan_after": int(info["makespan_so_far"]),
            "chosen_estimated_makespan_after": int(chosen_effect["estimated_makespan_after"]),
            "all_options": action_effects,
            "not_chosen_options": not_chosen,
            "guidance_applied": bool(step_idx in guidance_map),
            "decision_source": (
                "replay" if is_replay_prefix else ("guided" if step_idx in guidance_map else "model")
            ),
        }
        raw_step_outputs.append(
            step_record
        )
        if print_step_trace:
            _print_step_trace([step_record])

    event_log = env.get_event_log()
    solution_lines = ["Solution:", ""]
    for op in event_log:
        solution_lines.append(
            f"Job {op['job_id']} Operation {op['op_idx']}, M{op['machine_id']}"
        )
    solution_lines.append("")
    solution_lines.append(f"Makespan: {env.get_makespan()}")
    solution_text = "\n".join(solution_lines)

    return {
        "solution_text": solution_text,
        "makespan": env.get_makespan(),
        "raw_step_outputs": raw_step_outputs,
    }

def run_step_rollout(
    model,
    tokenizer,
    inst_for_ortools,
    env_mode: str = "serial",
    use_masking: bool = True,
    step_action_max_new_tokens: int = 8,
    include_problem_context: bool = True,
    enable_step_improvement: bool = False,
    step_reflection_passes: int = 1,
    step_reflection_topk: int = 3,
    emit_step_rationale: bool = False,
    step_rationale_max_new_tokens: int = 96,
    reason_topk: int = 3,
    use_reason_adapter: bool = False,
    action_code_width: int = 4,
    action_code_seed: int = 42,
    action_code_cap: int = 9999,
    print_step_trace: bool = False,
):
    best_result = _run_single_step_rollout(
        model=model,
        tokenizer=tokenizer,
        inst_for_ortools=inst_for_ortools,
        env_mode=env_mode,
        use_masking=use_masking,
        step_action_max_new_tokens=step_action_max_new_tokens,
        include_problem_context=include_problem_context,
        enable_step_improvement=False,
        step_reflection_passes=0,
        emit_step_rationale=emit_step_rationale,
        step_rationale_max_new_tokens=step_rationale_max_new_tokens,
        reason_topk=reason_topk,
        use_reason_adapter=use_reason_adapter,
        action_code_width=action_code_width,
        action_code_seed=action_code_seed,
        action_code_cap=action_code_cap,
        guidance_by_step=None,
        reflection_memory_text=None,
        print_step_trace=print_step_trace,
    )
    base_result = copy.deepcopy(best_result)
    best_result["base_result"] = base_result
    best_result["base_makespan"] = int(base_result["makespan"])
    best_result["final_makespan"] = int(best_result["makespan"])
    best_result["improvement_delta"] = 0
    best_result["improvement_enabled"] = bool(enable_step_improvement)
    best_result["improved_over_base"] = False
    best_result["improvement_history"] = []

    if not enable_step_improvement or int(step_reflection_passes) <= 0:
        return best_result

    notes = []
    improvement_history = []
    if print_step_trace:
        print(
            f"[Episode Improvement] start: passes={int(step_reflection_passes)}, "
            f"topk={max(1, int(step_reflection_topk))}, "
            f"baseline_makespan={int(best_result['makespan'])}"
        )
    for pass_idx in range(int(step_reflection_passes)):
        critical_steps = _select_top_critical_steps(
            best_result["raw_step_outputs"],
            top_k=max(1, int(step_reflection_topk)),
        )
        if not critical_steps:
            notes.append(f"pass {pass_idx + 1}: no critical step found")
            improvement_history.append(
                {
                    "pass_idx": int(pass_idx + 1),
                    "baseline_makespan": int(best_result["makespan"]),
                    "candidate_makespan": None,
                    "improved": False,
                    "guided_steps": [],
                    "reason": "no critical step found",
                }
            )
            if print_step_trace:
                print(
                    f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)}: "
                    "no critical step found"
                )
            break

        reflection_memory = _build_reflection_memory(
            current_makespan=int(best_result["makespan"]),
            critical_steps=critical_steps,
        )
        guidance_map = {}
        for critical_step in critical_steps:
            step_idx = int(critical_step["step_idx"])
            feasible_action_codes = list(critical_step["feasible_action_codes"])
            action_code_to_job = dict(critical_step["action_code_to_job"])
            chosen_code = str(critical_step["chosen_action_code"])
            chosen_job = int(critical_step["chosen_job"])
            best_alt = _best_alternative_option(critical_step)
            if print_step_trace:
                print(
                    f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)}: "
                    f"critical_step={step_idx}, chosen={chosen_code}"
                )

            suggested_code = None
            if feasible_action_codes:
                step_stack = resolve_step_stack(_normalize_env_mode(env_mode))
                improvement_prompt = step_stack.build_step_improvement_prompt(
                    state_text=critical_step["state_text"],
                    candidate_action_text=f"{chosen_code}",
                    feasible_jobs=feasible_action_codes,
                    reflection_memory=reflection_memory,
                    step_diagnostics=_build_step_diagnostics(critical_step),
                )
                improvement_prompt = (
                    f"{improvement_prompt}\n"
                    f"Episode summary: final makespan={best_result['makespan']}.\n"
                    f"Target step={step_idx}, chosen={chosen_code}.\n"
                    "Select an action that reduces bottleneck risk and final makespan."
                )
                reflection_prompt = _build_step_improvement_chat_prompt(
                    tokenizer, improvement_prompt
                )
                try:
                    suggested_code, _ = _sample_step_action(
                        model=model,
                        tokenizer=tokenizer,
                        prompt_text=reflection_prompt,
                        feasible_action_codes=feasible_action_codes,
                        use_masking=use_masking,
                        max_new_tokens=step_action_max_new_tokens,
                        action_code_width=action_code_width,
                        action_code_cap=action_code_cap,
                    )
                except StepActionParseError as exc:
                    notes.append(
                        f"pass {pass_idx + 1} step {step_idx}: improvement parse failed ({exc})"
                    )
                    if print_step_trace:
                        print(
                            f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)} "
                            f"step {step_idx}: parse failed ({exc})"
                        )

            if (not suggested_code or str(suggested_code) == chosen_code) and best_alt is not None:
                alt_code = str(best_alt["action_code"])
                if alt_code in action_code_to_job and alt_code != chosen_code:
                    suggested_code = alt_code
                    notes.append(
                        f"pass {pass_idx + 1} step {step_idx}: deterministic critic suggested {alt_code}"
                    )
                    if print_step_trace:
                        print(
                            f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)} "
                            f"step {step_idx}: critic suggested {alt_code}"
                        )

            if not suggested_code:
                continue
            suggested_job = int(action_code_to_job[suggested_code])
            if suggested_job == chosen_job:
                continue

            if print_step_trace:
                print(
                    f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)} "
                    f"step {step_idx}: suggested={suggested_code} -> Job {suggested_job}"
                )
            guidance_map[step_idx] = {
                "preferred_job": int(suggested_job),
                "avoid_jobs": [int(chosen_job)],
                "reason": (
                    f"chosen={chosen_code}/Job{chosen_job} showed high bottleneck risk; "
                    f"prefer {suggested_code}/Job{suggested_job}"
                ),
                "suggested_action_code": str(suggested_code),
            }

        if not guidance_map:
            notes.append(f"pass {pass_idx + 1}: no actionable guidance generated")
            improvement_history.append(
                {
                    "pass_idx": int(pass_idx + 1),
                    "baseline_makespan": int(best_result["makespan"]),
                    "candidate_makespan": None,
                    "improved": False,
                    "guided_steps": [],
                    "reason": "no actionable guidance generated",
                }
            )
            if print_step_trace:
                print(
                    f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)}: "
                    "no actionable guidance generated"
                )
            continue

        replay_action_jobs_by_step = {
            int(step["step_idx"]): int(step["chosen_job"])
            for step in best_result["raw_step_outputs"]
        }
        replay_prefix_until = min(int(step_idx) for step_idx in guidance_map.keys())
        if print_step_trace:
            print(
                f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)}: "
                f"replay_prefix_until={int(replay_prefix_until)} "
                "(prefix actions replayed, tail regenerated)"
            )

        candidate_result = _run_single_step_rollout(
            model=model,
            tokenizer=tokenizer,
            inst_for_ortools=inst_for_ortools,
            env_mode=env_mode,
            use_masking=use_masking,
            step_action_max_new_tokens=step_action_max_new_tokens,
            include_problem_context=include_problem_context,
            enable_step_improvement=False,
            step_reflection_passes=0,
            emit_step_rationale=emit_step_rationale,
            step_rationale_max_new_tokens=step_rationale_max_new_tokens,
            reason_topk=reason_topk,
            use_reason_adapter=use_reason_adapter,
            action_code_width=action_code_width,
            action_code_seed=action_code_seed + (pass_idx + 1) * 9973,
            action_code_cap=action_code_cap,
            guidance_by_step=guidance_map,
            reflection_memory_text=reflection_memory,
            replay_action_jobs_by_step=replay_action_jobs_by_step,
            replay_prefix_until=replay_prefix_until,
            print_step_trace=False,
        )
        improvement_history.append(
            {
                "pass_idx": int(pass_idx + 1),
                "baseline_makespan": int(best_result["makespan"]),
                "candidate_makespan": int(candidate_result["makespan"]),
                "improved": int(candidate_result["makespan"]) < int(best_result["makespan"]),
                "guided_steps": sorted(int(x) for x in guidance_map.keys()),
                "replay_prefix_until": int(replay_prefix_until),
                "reflection_memory": reflection_memory,
            }
        )
        if int(candidate_result["makespan"]) < int(best_result["makespan"]):
            notes.append(
                f"pass {pass_idx + 1}: improved {best_result['makespan']} -> {candidate_result['makespan']} "
                f"with guided_steps={sorted(guidance_map.keys())}"
            )
            if print_step_trace:
                print(
                    f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)}: "
                    f"improved {int(best_result['makespan'])} -> {int(candidate_result['makespan'])}"
                )
            best_result = candidate_result
        else:
            notes.append(
                f"pass {pass_idx + 1}: no improvement ({best_result['makespan']} vs {candidate_result['makespan']})"
            )
            if print_step_trace:
                print(
                    f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)}: "
                    f"no improvement ({int(best_result['makespan'])} vs {int(candidate_result['makespan'])})"
                )

    best_result["base_result"] = base_result
    best_result["base_makespan"] = int(base_result["makespan"])
    best_result["final_makespan"] = int(best_result["makespan"])
    best_result["improvement_delta"] = int(base_result["makespan"]) - int(best_result["makespan"])
    best_result["improved_over_base"] = int(best_result["makespan"]) < int(base_result["makespan"])
    best_result["improvement_enabled"] = True
    best_result["episode_improvement_notes"] = notes
    best_result["improvement_history"] = improvement_history
    if print_step_trace:
        _print_step_trace(best_result["raw_step_outputs"])
        if notes:
            print("episode_improvement_notes:")
            for x in notes:
                print(" -", x)
    return best_result

def build_machine_log_from_step_outputs(raw_step_outputs):
    rows = []
    for step in raw_step_outputs or []:
        st = int(step.get("chosen_start_time", 0))
        ed = int(step.get("chosen_end_time", st))
        rows.append(
            {
                "Machine": int(step.get("machine_id", -1)),
                "Start": st,
                "End": ed,
                "Delta": int(max(0, ed - st)),
                "JobNum": int(step.get("chosen_job", -1)),
                "Operation": int(step.get("op_idx", -1)),
                "ActionCode": str(step.get("chosen_action_code", "")),
                "Reason": str(step.get("rationale_text", "")),
            }
        )
    return rows

def save_result_csv_artifacts(
    result,
    output_dir,
    idx,
    n,
    m,
    benchmark_makespan=None,
    example_path="",
    prefix="eval",
    sample_idx=None,
    variant="final",
    extra_summary=None,
):
    artifact_dir = Path(output_dir) / "per_instance_csv"
    artifact_dir.mkdir(parents=True, exist_ok=True)

    stem_parts = [f"{prefix}_{int(idx):04d}"]
    if sample_idx is not None:
        stem_parts.append(f"sample_{int(sample_idx) + 1:02d}")
    if variant:
        stem_parts.append(str(variant))
    stem = "_".join(stem_parts)
    step_csv_path = artifact_dir / f"{stem}_steps.csv"
    machine_csv_path = artifact_dir / f"{stem}_machine_log.csv"
    summary_csv_path = artifact_dir / f"{stem}_summary.csv"

    raw_steps = list(result.get("raw_step_outputs", []) or [])
    step_fields = sorted({k for row in raw_steps for k in row.keys()})
    with open(step_csv_path, mode="w", newline="", encoding="utf-8-sig") as f:
        if step_fields:
            w = csv.DictWriter(f, fieldnames=step_fields)
            w.writeheader()
            for row in raw_steps:
                w.writerow({k: row.get(k) for k in step_fields})

    machine_rows = build_machine_log_from_step_outputs(raw_steps)
    machine_fields = [
        "Machine",
        "Start",
        "End",
        "Delta",
        "JobNum",
        "Operation",
        "ActionCode",
        "Reason",
    ]
    with open(machine_csv_path, mode="w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=machine_fields)
        w.writeheader()
        for row in machine_rows:
            w.writerow(row)

    makespan = int(result.get("makespan", 0))
    gap = None
    if benchmark_makespan not in (None, 0):
        gap = abs((makespan - float(benchmark_makespan)) / float(benchmark_makespan))

    summary_fields = [
        "Index",
        "Num_J",
        "Num_M",
        "SampleIndex",
        "Variant",
        "Actual_Makespan",
        "Benchmark_Makespan",
        "Gap",
        "Num_Steps",
        "ImprovementEnabled",
        "ImprovedOverBase",
        "Base_Makespan",
        "Final_Makespan",
        "ImprovementDelta",
        "WasBest",
        "Path",
        "Prefix",
    ]
    if extra_summary:
        for key in extra_summary.keys():
            if key not in summary_fields:
                summary_fields.append(key)
    with open(summary_csv_path, mode="w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=summary_fields)
        w.writeheader()
        summary_row = {
            "Index": int(idx),
            "Num_J": int(n),
            "Num_M": int(m),
            "SampleIndex": None if sample_idx is None else int(sample_idx),
            "Variant": str(variant),
            "Actual_Makespan": makespan,
            "Benchmark_Makespan": benchmark_makespan,
            "Gap": gap,
            "Num_Steps": len(raw_steps),
            "ImprovementEnabled": int(bool(result.get("improvement_enabled", False))),
            "ImprovedOverBase": int(bool(result.get("improved_over_base", False))),
            "Base_Makespan": result.get("base_makespan"),
            "Final_Makespan": result.get("final_makespan", makespan),
            "ImprovementDelta": result.get("improvement_delta"),
            "WasBest": int(bool((extra_summary or {}).get("WasBest", False))),
            "Path": example_path,
            "Prefix": prefix,
        }
        if extra_summary:
            summary_row.update(extra_summary)
        w.writerow(summary_row)

    return {
        "step_csv": str(step_csv_path),
        "machine_csv": str(machine_csv_path),
        "summary_csv": str(summary_csv_path),
    }

def build_sequence_trace_rows(
    result,
    idx,
    n,
    m,
    benchmark_makespan=None,
    example_path="",
    sample_idx=None,
    variant="final",
    was_best=False,
):
    rows = []
    raw_steps = list(result.get("raw_step_outputs", []) or [])
    for step in raw_steps:
        rows.append(
            {
                "Index": int(idx),
                "SampleIndex": None if sample_idx is None else int(sample_idx),
                "Variant": str(variant),
                "WasBest": int(bool(was_best)),
                "Num_J": int(n),
                "Num_M": int(m),
                "Benchmark_Makespan": benchmark_makespan,
                "Actual_Makespan": int(result.get("makespan", 0)),
                "ImprovementEnabled": int(bool(result.get("improvement_enabled", False))),
                "ImprovedOverBase": int(bool(result.get("improved_over_base", False))),
                "Base_Makespan": result.get("base_makespan"),
                "Final_Makespan": result.get("final_makespan", result.get("makespan")),
                "ImprovementDelta": result.get("improvement_delta"),
                "Path": example_path,
                "StepIdx": int(step.get("step_idx", -1)),
                "DecisionSource": str(step.get("decision_source", "")),
                "GuidanceApplied": int(bool(step.get("guidance_applied", False))),
                "ChosenAction": str(step.get("chosen_action_code", "")),
                "ChosenJob": int(step.get("chosen_job", -1)),
                "Operation": int(step.get("op_idx", -1)),
                "Machine": int(step.get("machine_id", -1)),
                "Start": int(step.get("chosen_start_time", 0)),
                "End": int(step.get("chosen_end_time", 0)),
                "ProcTime": int(step.get("chosen_proc_time", 0)),
                "MakespanBefore": int(step.get("makespan_before", 0)),
                "MakespanAfter": int(step.get("makespan_after", 0)),
                "EstimatedMakespanAfter": int(step.get("chosen_estimated_makespan_after", 0)),
                "NumFeasibleActions": len(list(step.get("feasible_action_codes", []) or [])),
                "ModelOutput": str(step.get("model_output", "")),
                "Rationale": str(step.get("rationale_text", "")),
            }
        )
    return rows

def append_sequence_trace_rows(sequence_trace_csv_filename, rows):
    if not sequence_trace_csv_filename or not rows:
        return
    fieldnames = [
        "Index",
        "SampleIndex",
        "Variant",
        "WasBest",
        "Num_J",
        "Num_M",
        "Benchmark_Makespan",
        "Actual_Makespan",
        "ImprovementEnabled",
        "ImprovedOverBase",
        "Base_Makespan",
        "Final_Makespan",
        "ImprovementDelta",
        "Path",
        "StepIdx",
        "DecisionSource",
        "GuidanceApplied",
        "ChosenAction",
        "ChosenJob",
        "Operation",
        "Machine",
        "Start",
        "End",
        "ProcTime",
        "MakespanBefore",
        "MakespanAfter",
        "EstimatedMakespanAfter",
        "NumFeasibleActions",
        "ModelOutput",
        "Rationale",
    ]
    write_header = not os.path.exists(sequence_trace_csv_filename)
    with open(sequence_trace_csv_filename, mode="a", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        for row in rows:
            writer.writerow({k: row.get(k) for k in fieldnames})

def evaluate_model_step(
    model,
    tokenizer,
    dataset,
    run_args=None,
    cfg=None,
    csv_filename="evaluation_results_step.csv",
    aggregate_csv_filename=None,
    sequence_trace_csv_filename=None,
    num_return_sequences=1,
    use_masking=True,
):
    run_args = _coerce_run_args(run_args=run_args, cfg=cfg)
    env_mode = _normalize_env_mode(getattr(run_args, "env_mode", "serial"))
    all_predictions = []
    all_references = []

    write_header = not os.path.exists(csv_filename)
    with open(csv_filename, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        if write_header:
            writer.writerow(
                [
                    "Index",
                    "Num_J",
                    "Num_M",
                    "Declared_makespan",
                    "Actual_Makespan",
                    "Gap",
                    "Total_Sampling_Attempts",
                    "Num_Feasible_Solutions",
                    "Feasible_Sol_Makespans",
                    "Validation_Message",
                    "Path",
                    "Best_Solution",
                    "Avg_Time",
                ]
            )

    if aggregate_csv_filename is None:
        aggregate_csv_filename = str(
            Path(csv_filename).with_name("evaluate_all_problems.csv")
        )
    sample_count = max(int(num_return_sequences), 1)
    aggregate_fieldnames = [
        "Index",
        "Num_J",
        "Num_M",
        "Benchmark_Makespan",
        "Gap",
        "Best_Gap",
        "Best_Makespan",
        "Mean_Makespan",
        "Var_Makespan",
        "Std_Makespan",
        "Num_Attempts",
        "Num_Feasible",
        "Feasible_Rate",
        "Path",
    ]
    for k in range(sample_count):
        aggregate_fieldnames.append(f"Sample{k + 1}_Feasible")
        aggregate_fieldnames.append(f"Sample{k + 1}_Makespan")
        aggregate_fieldnames.append(f"Sample{k + 1}_BaseMakespan")
        aggregate_fieldnames.append(f"Sample{k + 1}_Improved")
        aggregate_fieldnames.append(f"Sample{k + 1}_ImprovementDelta")

    aggregate_write_header = not os.path.exists(aggregate_csv_filename)
    with open(aggregate_csv_filename, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=aggregate_fieldnames)
        if aggregate_write_header:
            writer.writeheader()

    for idx, example in tqdm(enumerate(dataset["train"])):
        matrix_content = example["matrix"]
        n, m, inst_for_ortools, real_makespan = read_matrix_form_jssp(matrix_content)
        ms = real_makespan if real_makespan is not None else example.get("makespan")

        rollout_results = []
        sample_feasible = []
        sample_makespans = []
        total_time = 0.0
        for _ in range(sample_count):
            start = time.time()
            result = run_step_rollout(
                model=model,
                tokenizer=tokenizer,
                inst_for_ortools=inst_for_ortools,
                env_mode=env_mode,
                use_masking=use_masking,
                step_action_max_new_tokens=run_args.step_action_max_new_tokens,
                include_problem_context=not run_args.disable_step_problem_context,
                enable_step_improvement=run_args.enable_step_improvement,
                step_reflection_passes=run_args.step_reflection_passes,
                step_reflection_topk=run_args.step_reflection_topk,
                emit_step_rationale=run_args.emit_step_rationale,
                step_rationale_max_new_tokens=run_args.step_rationale_max_new_tokens,
                reason_topk=run_args.reason_topk,
                use_reason_adapter=bool(run_args.reason_model_path),
                action_code_width=run_args.action_code_width,
                action_code_seed=run_args.action_code_seed + idx,
                action_code_cap=run_args.action_code_cap,
                print_step_trace=run_args.print_step_trace,
            )
            total_time += time.time() - start
            rollout_results.append(result)
            sample_feasible.append(True)
            sample_makespans.append(int(result["makespan"]))

        feasible_results = [r for r in rollout_results if r is not None]
        feasible_makespans = [int(r["makespan"]) for r in feasible_results]

        if feasible_results:
            best_idx = int(np.argmin(feasible_makespans))
            best_result = feasible_results[best_idx]
            declared_makespan = int(best_result["makespan"])
            best_solution = best_result["solution_text"]
        else:
            best_result = None
            declared_makespan = None
            best_solution = "NO_FEASIBLE_SOLUTION"

        gap = None
        if declared_makespan is not None and ms and ms > 0:
            gap = abs((declared_makespan - ms) / ms)

        all_predictions.append(best_solution)
        avg_time = total_time / sample_count
        if declared_makespan is not None:
            validation_message = f"Step rollout feasible with makespan {declared_makespan}"
        else:
            validation_message = "No feasible rollout"

        with open(csv_filename, mode="a", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(
                [
                    idx,
                    n,
                    m,
                    declared_makespan,
                    ms,
                    gap,
                    len(rollout_results),
                    len(feasible_results),
                    feasible_makespans,
                    validation_message,
                    example.get("path", ""),
                    " ",
                    avg_time,
                ]
            )

        output_root = run_args.output_dir or str(Path(csv_filename).parent)
        if best_result is not None:
            saved_paths = save_result_csv_artifacts(
                result=best_result,
                output_dir=output_root,
                idx=idx,
                n=n,
                m=m,
                benchmark_makespan=ms,
                example_path=example.get("path", ""),
                prefix="eval",
                variant="best",
                extra_summary={"WasBest": True},
            )
            print(f"[saved csv] idx={idx}: {saved_paths['step_csv']}")

        feasible_values = [v for v in sample_makespans if v is not None]
        if feasible_values:
            mean_ms = float(np.mean(feasible_values))
            var_ms = float(np.var(feasible_values))
            std_ms = float(np.std(feasible_values))
        else:
            mean_ms = None
            var_ms = None
            std_ms = None

        agg_row = {
            "Index": idx,
            "Num_J": int(n),
            "Num_M": int(m),
            "Benchmark_Makespan": ms,
            "Gap": gap,
            "Best_Gap": gap,
            "Best_Makespan": declared_makespan,
            "Mean_Makespan": mean_ms,
            "Var_Makespan": var_ms,
            "Std_Makespan": std_ms,
            "Num_Attempts": sample_count,
            "Num_Feasible": len(feasible_values),
            "Feasible_Rate": float(len(feasible_values) / sample_count),
            "Path": example.get("path", ""),
        }
        for k in range(sample_count):
            agg_row[f"Sample{k + 1}_Feasible"] = int(bool(sample_feasible[k]))
            agg_row[f"Sample{k + 1}_Makespan"] = sample_makespans[k]
            sample_result = rollout_results[k]
            base_makespan = None if sample_result is None else sample_result.get("base_makespan")
            agg_row[f"Sample{k + 1}_BaseMakespan"] = base_makespan
            agg_row[f"Sample{k + 1}_Improved"] = (
                None if sample_result is None else int(bool(sample_result.get("improved_over_base", False)))
            )
            agg_row[f"Sample{k + 1}_ImprovementDelta"] = (
                None if sample_result is None else sample_result.get("improvement_delta")
            )

        with open(aggregate_csv_filename, mode="a", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=aggregate_fieldnames)
            writer.writerow(agg_row)

        for sample_idx, result in enumerate(rollout_results):
            if result is None:
                continue
            is_best_sample = bool(
                best_result is not None and int(result["makespan"]) == int(best_result["makespan"])
            )
            save_result_csv_artifacts(
                result=result,
                output_dir=output_root,
                idx=idx,
                n=n,
                m=m,
                benchmark_makespan=ms,
                example_path=example.get("path", ""),
                prefix="eval",
                sample_idx=sample_idx,
                variant="final",
                extra_summary={"WasBest": is_best_sample},
            )
            append_sequence_trace_rows(
                sequence_trace_csv_filename,
                build_sequence_trace_rows(
                    result=result,
                    idx=idx,
                    n=n,
                    m=m,
                    benchmark_makespan=ms,
                    example_path=example.get("path", ""),
                    sample_idx=sample_idx,
                    variant="final",
                    was_best=is_best_sample,
                ),
            )
            base_result = result.get("base_result")
            if base_result is not None and bool(result.get("improvement_enabled", False)):
                save_result_csv_artifacts(
                    result=base_result,
                    output_dir=output_root,
                    idx=idx,
                    n=n,
                    m=m,
                    benchmark_makespan=ms,
                    example_path=example.get("path", ""),
                    prefix="eval",
                    sample_idx=sample_idx,
                    variant="base",
                    extra_summary={"WasBest": False},
                )
                append_sequence_trace_rows(
                    sequence_trace_csv_filename,
                    build_sequence_trace_rows(
                        result=base_result,
                        idx=idx,
                        n=n,
                        m=m,
                        benchmark_makespan=ms,
                        example_path=example.get("path", ""),
                        sample_idx=sample_idx,
                        variant="base",
                        was_best=False,
                    ),
                )

    return all_predictions, all_references

def run_demo_instance_step(
    model,
    tokenizer,
    dataset,
    run_args=None,
    cfg=None,
    index: int = 0,
    num_solutions: int = 1,
    use_masking: bool = True,
):
    run_args = _coerce_run_args(run_args=run_args, cfg=cfg)
    env_mode = _normalize_env_mode(getattr(run_args, "env_mode", "serial"))
    if "train" not in dataset:
        raise ValueError("Dataset must contain a 'train' split for demo runs.")

    dataset_length = len(dataset["train"])
    if index < 0 or index >= dataset_length:
        raise ValueError(f"demo_index {index} out of range (dataset has {dataset_length} items).")

    example = dataset["train"][index]
    matrix_content = example["matrix"]
    n, m, inst_for_ortools, real_makespan = read_matrix_form_jssp(matrix_content)
    ms = real_makespan if real_makespan is not None else example.get("makespan")

    print("=" * 80)
    print(
        f"🎯 Step-demo instance {index} — Jobs={n}, Machines={m}, "
        f"Masking={use_masking}, Context={not run_args.disable_step_problem_context}, "
        f"EpisodeImprovement={run_args.enable_step_improvement}, "
        f"Rationale={run_args.emit_step_rationale}"
    )
    print("=" * 80)

    best_result = None
    best_makespan = None
    demo_results = []
    for sample_idx in range(max(num_solutions, 1)):
        result = run_step_rollout(
            model=model,
            tokenizer=tokenizer,
            inst_for_ortools=inst_for_ortools,
            env_mode=env_mode,
            use_masking=use_masking,
            step_action_max_new_tokens=run_args.step_action_max_new_tokens,
            include_problem_context=not run_args.disable_step_problem_context,
            enable_step_improvement=run_args.enable_step_improvement,
            step_reflection_passes=run_args.step_reflection_passes,
            step_reflection_topk=run_args.step_reflection_topk,
            emit_step_rationale=run_args.emit_step_rationale,
            step_rationale_max_new_tokens=run_args.step_rationale_max_new_tokens,
            reason_topk=run_args.reason_topk,
            use_reason_adapter=bool(run_args.reason_model_path),
            action_code_width=run_args.action_code_width,
            action_code_seed=run_args.action_code_seed + sample_idx,
            action_code_cap=run_args.action_code_cap,
            print_step_trace=run_args.print_step_trace,
        )
        demo_results.append(result)
        makespan = int(result["makespan"])
        print(f"\n--- Step rollout {sample_idx + 1}/{num_solutions} ---")
        print(result["solution_text"])
        print(f"makespan={makespan}")
        if not run_args.print_step_trace:
            _print_step_trace(result.get("raw_step_outputs", []))
        notes = result.get("episode_improvement_notes")
        if notes and not run_args.print_step_trace:
            print("episode_improvement_notes:")
            for x in notes:
                print(" -", x)
        if run_args.emit_step_rationale:
            print("step rationales:")
            for step_info in result.get("raw_step_outputs", []):
                rationale = step_info.get("rationale_text")
                if rationale:
                    print(
                        f"  - step {step_info['step_idx']}: "
                        f"{step_info.get('chosen_action_code')} (job {step_info['chosen_job']}) -> {rationale}"
                    )

        if best_makespan is None or makespan < best_makespan:
            best_makespan = makespan
            best_result = result

    if ms:
        print(f"\nBenchmark makespan: {ms}")
        if best_makespan is not None and ms > 0:
            print(f"Best gap: {abs((best_makespan - ms) / ms):.4f}")

    if best_result is not None:
        output_root = run_args.output_dir or "val_results/jssp_val"
        demo_saved_paths = save_result_csv_artifacts(
            result=best_result,
            output_dir=output_root,
            idx=index,
            n=n,
            m=m,
            benchmark_makespan=ms,
            example_path=example.get("path", ""),
            prefix="demo",
            variant="best",
            extra_summary={"WasBest": True},
        )
        print("demo csv saved:", demo_saved_paths)
        for sample_idx, result in enumerate(demo_results):
            is_best_sample = bool(int(result["makespan"]) == int(best_result["makespan"]))
            save_result_csv_artifacts(
                result=result,
                output_dir=output_root,
                idx=index,
                n=n,
                m=m,
                benchmark_makespan=ms,
                example_path=example.get("path", ""),
                prefix="demo",
                sample_idx=sample_idx,
                variant="final",
                extra_summary={"WasBest": is_best_sample},
            )
            base_result = result.get("base_result")
            if base_result is not None and bool(result.get("improvement_enabled", False)):
                save_result_csv_artifacts(
                    result=base_result,
                    output_dir=output_root,
                    idx=index,
                    n=n,
                    m=m,
                    benchmark_makespan=ms,
                    example_path=example.get("path", ""),
                    prefix="demo",
                    sample_idx=sample_idx,
                    variant="base",
                    extra_summary={"WasBest": False},
                )

    print("✅ Step demo run complete.")
    return best_result

activate_adapter_if_available = _activate_adapter_if_available
maybe_load_reason_adapter = _maybe_load_reason_adapter
sample_step_action = _sample_step_action
sample_step_rationale = _sample_step_rationale
emit_step_trace = _print_step_trace
run_single_step_rollout = _run_single_step_rollout


def run_step_rollout_from_cfg(model, tokenizer, inst_for_ortools, cfg):
    return run_step_rollout(
        model=model,
        tokenizer=tokenizer,
        inst_for_ortools=inst_for_ortools,
        env_mode=str(cfg.get('env_mode', 'serial')),
        use_masking=not bool(cfg.get('disable_masking', False)),
        step_action_max_new_tokens=int(cfg.get('step_action_max_new_tokens', 8)),
        include_problem_context=not bool(cfg.get('disable_step_problem_context', False)),
        enable_step_improvement=bool(cfg.get('enable_step_improvement', False)),
        step_reflection_passes=int(cfg.get('step_reflection_passes', 1)),
        step_reflection_topk=int(cfg.get('step_reflection_topk', 3)),
        emit_step_rationale=bool(cfg.get('emit_step_rationale', False)),
        step_rationale_max_new_tokens=int(cfg.get('step_rationale_max_new_tokens', 96)),
        reason_topk=int(cfg.get('reason_topk', 3)),
        use_reason_adapter=bool(cfg.get('enable_reason_adapter', False)),
        action_code_width=int(cfg.get('action_code_width', 4)),
        action_code_seed=int(cfg.get('action_code_seed', 42)),
        action_code_cap=int(cfg.get('action_code_cap', 9999)),
        print_step_trace=bool(cfg.get('print_step_trace', False)),
    )


## 3) 모델/데이터 로드 + 실행


In [4]:
MODEL_BASE = {
    'llama8b': 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    'llama1b': 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit',
    'qwen2.5_7b': 'unsloth/Qwen2.5-7B-Instruct-unsloth-bnb-4bit',
    'qwen2.5_14b': 'unsloth/Qwen2.5-14B-Instruct-unsloth-bnb-4bit',
    'deepseek_8b': 'unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit',
    'qwen25_7b_math': 'unsloth/Qwen2.5-Math-7B-Instruct-bnb-4bit',
}
print('base model family:', MODEL_BASE[CFG['model_type']])

ACTION_TOKEN_PREFIX = "A"

def format_action_code(code_id: int, code_width: int = 4):
    return f"<{ACTION_TOKEN_PREFIX}{int(code_id):0{int(code_width)}d}>"

def build_action_special_tokens(code_width: int = 4, code_cap: int = 9999):
    return [format_action_code(i, code_width=code_width) for i in range(1, int(code_cap) + 1)]

def ensure_action_special_tokens(tokenizer, model, code_width: int = 4, code_cap: int = 9999):
    action_tokens = build_action_special_tokens(code_width=code_width, code_cap=code_cap)
    current_vocab = tokenizer.get_vocab()
    to_add = [tok for tok in action_tokens if tok not in current_vocab]
    added = 0
    if to_add:
        added = tokenizer.add_special_tokens({'additional_special_tokens': to_add})
        model.resize_token_embeddings(len(tokenizer))
    return {
        'num_added_tokens': int(added),
        'vocab_size': int(len(tokenizer)),
        'action_token_count': int(len(action_tokens)),
    }

def resolve_checkpoint_tag(repo_id: str, checkpoint_tag: str | None, token: str | None = None):
    if not checkpoint_tag:
        return None
    repo_id = os.path.expanduser(str(repo_id))
    if os.path.exists(repo_id):
        return None
    checkpoint_tag = str(checkpoint_tag)
    if checkpoint_tag != 'latest_checkpoint':
        return checkpoint_tag
    try:
        api = HfApi(token=token)
        files = api.list_repo_files(repo_id=repo_id, repo_type='model')
    except Exception:
        return None
    checkpoint_dirs = sorted(
        {
            path.split('/')[0]
            for path in files
            if path.startswith('checkpoint-') and '/' in path
        },
        key=lambda name: int(name.split('-')[-1]),
    )
    return checkpoint_dirs[-1] if checkpoint_dirs else None


def resolve_model_source(repo_or_path: str | None, local_path: str | None = None, checkpoint_tag: str | None = None, token: str | None = None):
    if local_path:
        return os.path.expanduser(local_path)
    if not repo_or_path:
        return None
    repo_or_path = os.path.expanduser(repo_or_path)
    if os.path.exists(repo_or_path):
        return repo_or_path
    resolved_checkpoint = resolve_checkpoint_tag(repo_or_path, checkpoint_tag, token=token)
    if resolved_checkpoint is None:
        return repo_or_path

    snapshot_dir = snapshot_download(
        repo_id=repo_or_path,
        repo_type='model',
        allow_patterns=[f'{resolved_checkpoint}/*'],
        token=token,
    )
    local_checkpoint_dir = os.path.join(snapshot_dir, resolved_checkpoint)
    if os.path.exists(local_checkpoint_dir):
        return local_checkpoint_dir

    for probe_file in ('adapter_config.json', 'adapter_model.safetensors'):
        try:
            local_file = hf_hub_download(
                repo_id=repo_or_path,
                repo_type='model',
                filename=f'{resolved_checkpoint}/{probe_file}',
                token=token,
            )
            return os.path.dirname(local_file)
        except Exception:
            pass

    raise FileNotFoundError(
        f'checkpoint folder not found after download: {local_checkpoint_dir}'
    )


def is_adapter_source(path_or_repo: str | None, token: str | None = None) -> bool:
    if not path_or_repo:
        return False
    if os.path.exists(path_or_repo):
        return os.path.exists(os.path.join(path_or_repo, 'adapter_config.json'))
    try:
        api = HfApi(token=token)
        files = set(api.list_repo_files(repo_id=path_or_repo, repo_type='model'))
    except Exception:
        return False
    return 'adapter_config.json' in files


def maybe_load_policy_adapter(model, policy_model_path: str | None, token: str | None = None):
    if not policy_model_path:
        return False
    if not hasattr(model, 'load_adapter'):
        raise RuntimeError(
            'Policy adapter path was provided, but the loaded model does not support load_adapter().'
        )
    if os.path.exists(policy_model_path):
        model.load_adapter(policy_model_path, adapter_name='default', adapter_kwargs={'local_files_only': True})
    else:
        model.load_adapter(policy_model_path, adapter_name='default', token=token)
    if hasattr(model, 'set_adapter'):
        model.set_adapter('default')
    return True


model_path = resolve_model_source(
    CFG.get('model_repo_or_path'),
    local_path=CFG.get('model_path'),
    checkpoint_tag=CFG.get('model_checkpoint_tag'),
    token=CFG.get('hf_token'),
)
use_reason_adapter = bool(CFG.get('enable_reason_adapter', False) and CFG.get('emit_step_rationale', False))
reason_model_path = resolve_model_source(
    CFG.get('reason_model_repo_or_path'),
    local_path=CFG.get('reason_model_path'),
    checkpoint_tag=CFG.get('reason_model_checkpoint_tag'),
    token=CFG.get('hf_token'),
) if (use_reason_adapter and (CFG.get('reason_model_repo_or_path') or CFG.get('reason_model_path'))) else None
print('loading model from:', model_path)
print('use_reason_adapter:', use_reason_adapter)
if reason_model_path:
    print('loading reason adapter from:', reason_model_path)

dtype = torch.bfloat16 if CFG['dtype'] == 'bfloat16' else torch.float16
load_name = MODEL_BASE[CFG['model_type']] if is_adapter_source(model_path, token=CFG.get('hf_token')) else model_path
print('base load path:', load_name)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=load_name,
    max_seq_length=CFG['max_seq_length'],
    load_in_4bit=CFG['load_in_4bit'],
    dtype=dtype,
    local_files_only=False,
)
action_token_install = ensure_action_special_tokens(
    tokenizer=tokenizer,
    model=model,
    code_width=int(CFG.get('action_code_width', 4)),
    code_cap=int(CFG.get('action_code_cap', 9999)),
)
print('action token install:', action_token_install)
if is_adapter_source(model_path, token=CFG.get('hf_token')):
    maybe_load_policy_adapter(model, model_path, token=CFG.get('hf_token'))
    print('policy adapter loaded')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('model device target:', device)
FastLanguageModel.for_inference(model)
if CFG.get('enable_reason_adapter', False) and reason_model_path:
    maybe_load_reason_adapter(model, reason_model_path)
    print('reason adapter loaded')

if CFG['eval_source'] == 'hf':
    eval_path = hf_hub_download(
        repo_id=CFG['eval_dataset_repo'],
        repo_type='dataset',
        filename=CFG['eval_dataset_file'],
        token=CFG['hf_token'],
    )
elif CFG['eval_source'] == 'local':
    eval_path = os.path.expanduser(CFG['eval_data_path'])
else:
    raise ValueError("CFG['eval_source'] must be 'hf' or 'local'")

print('eval dataset path:', eval_path)
eval_dataset = load_dataset('json', data_files=eval_path)

task_type = 'fssp' if CFG['infer_fssp'] else 'jssp'
output_style = 'accord' if CFG['output_accord'] else 'list_of_lists'
output_dir = Path(CFG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

model_base_name = os.path.basename(model_path) if model_path else 'model'
csv_filename = output_dir / f"output_step_{output_style}_{model_base_name}_num_return_sequences{CFG['num_return_sequences']}.csv"
aggregate_csv_filename = output_dir / f"evaluate_all_problems_{model_base_name}_num_return_sequences{CFG['num_return_sequences']}.csv"
sequence_trace_csv_filename = output_dir / f"evaluate_sequence_trace_{model_base_name}_num_return_sequences{CFG['num_return_sequences']}.csv"



base model family: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit


Fetching 0 files: 0it [00:00, ?it/s]

FileNotFoundError: checkpoint folder not found after download: /root/.cache/huggingface/hub/models--HYUNJINI--jssp_policy_llama8b_step_r64_ep2/snapshots/82a419ad31e90908c184b6a16398754de1f37f1d/checkpoint-33000

In [ ]:
print('step-by-step inference start')
print('masking enabled:', not CFG['disable_masking'])
print('csv path:', csv_filename)
print('aggregate csv path:', aggregate_csv_filename)
print('sequence trace csv path:', sequence_trace_csv_filename)

LAST_DEMO_RESULT = None
LAST_DEMO_META = None

if CFG['demo_index'] is not None:
    demo_num_solutions = CFG['demo_num_solutions'] if CFG['demo_num_solutions'] is not None else CFG['num_return_sequences']
    LAST_DEMO_RESULT = run_demo_instance_step(
        model=model,
        tokenizer=tokenizer,
        dataset=eval_dataset,
        cfg=CFG,
        index=int(CFG['demo_index']),
        num_solutions=int(demo_num_solutions),
    )
    LAST_DEMO_META = {
        'demo_index': int(CFG['demo_index']),
        'num_solutions': int(demo_num_solutions),
        'csv_filename': str(csv_filename),
    }
else:
    evaluate_model_step(
        model=model,
        tokenizer=tokenizer,
        dataset=eval_dataset,
        cfg=CFG,
        csv_filename=str(csv_filename),
        aggregate_csv_filename=str(aggregate_csv_filename),
        sequence_trace_csv_filename=str(sequence_trace_csv_filename),
    )
    print('inference complete')
    print('csv saved:', csv_filename)
    print('aggregate csv saved:', aggregate_csv_filename)
    print('sequence trace csv saved:', sequence_trace_csv_filename)




## 4) gantt

In [ ]:
# --- Independent Gantt Generator from saved machine CSV ---
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from pathlib import Path


def create_static_gantt(machine_log, makespan, save_path):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    if machine_log.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.text(0.5, 0.5, 'No valid schedule data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('JSSP Gantt Chart (No Valid Data)')
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        return

    unique_jobs = sorted(machine_log['JobNum'].dropna().unique().tolist())
    cmap = plt.cm.get_cmap('tab20', max(len(unique_jobs), 1))
    job_colors = {job: mcolors.rgb2hex(cmap(i % cmap.N)) for i, job in enumerate(unique_jobs)}
    machine_log = machine_log.copy()
    machine_log['color'] = machine_log['JobNum'].map(job_colors)

    fig, ax = plt.subplots(figsize=(16, 10))
    ax.barh(
        machine_log['Machine'],
        machine_log['Delta'],
        left=machine_log['Start'],
        color=machine_log['color'],
        edgecolor='black',
        linewidth=0.5,
        alpha=0.85,
    )

    for _, row in machine_log.iterrows():
        ax.text(
            row['Start'] + row['Delta'] / 2,
            row['Machine'],
            f"J{int(row['JobNum'])}O{int(row['Operation'])}",
            ha='center',
            va='center',
            fontsize=7,
            fontweight='bold',
        )

    legend_elements = [Patch(facecolor=job_colors[j], label=f'Job {j}') for j in unique_jobs]
    if legend_elements:
        ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.15, 1))

    ax.set_xlabel('Time')
    ax.set_ylabel('Machine')
    ax.set_title(f'JSSP Gantt Chart (Makespan: {int(makespan)})', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)

    limit = int(makespan) if int(makespan) > 0 else int(machine_log['End'].max())
    ax.set_xlim(0, max(limit, 1) * 1.05)
    ax.axvline(x=int(makespan), color='red', linestyle='--', linewidth=2, alpha=0.75)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


# usage: generate gantt for all saved machine CSV files
artifact_dir = Path(CFG.get('output_dir', '/content/val_results/jssp_val')) / 'per_instance_csv'
if not artifact_dir.exists():
    print('artifact dir not found:', artifact_dir)
else:
    machine_files = sorted(artifact_dir.glob('*_machine_log.csv'))
    if not machine_files:
        print('no machine csv found under:', artifact_dir)
    else:
        for machine_csv in machine_files:
            stem = machine_csv.name.replace('_machine_log.csv', '')
            summary_csv = artifact_dir / f"{stem}_summary.csv"
            machine_log = pd.read_csv(machine_csv)
            makespan = int(machine_log['End'].max()) if len(machine_log) else 0
            if summary_csv.exists():
                try:
                    s = pd.read_csv(summary_csv)
                    if 'Actual_Makespan' in s.columns and len(s):
                        makespan = int(float(s.iloc[0]['Actual_Makespan']))
                except Exception:
                    pass
            out_png = artifact_dir / f"{stem}_gantt.png"
            create_static_gantt(machine_log, makespan, out_png)
            print('gantt saved:', out_png)

